# RAG Quarterly Earnings Novelty Analysis 
#How to use? Change the ticker and company name and email and use Claude API.

Complete standalone pipeline:
1. Pull the last N earnings 8-K filings from SEC EDGAR
2. Clean and chunk the press release text
3. Derive quarter labels from filing dates (works for any ticker)
4. Embed every chunk with a sentence transformer
5. Score per-chunk novelty vs. the prior quarter (TREC-style)
6. Report the top novel chunks and summary stats

#References for getting inspirations: 
- Allan, Bolivar & Wade (2003)
- Saha, A., et al. (2025). BlackRock LLM agents survey.
- Ghafouri, Mohammadreza, et al. Risk, Ambiguity, and Infinity: Behavioral Signatures of Modern Large Language Models. 2025.
- Allan, James, Courtney Wade, and Alvaro Bolivar. Retrieval and Novelty Detection at the Sentence Level. 2003. 

# Actual implementations: 
- Karpukhin et al. (2020), EMNLP — Dense Passage Retrieval
- Soboroff & Harman (2005), NIST — TREC Novelty Track

## 1. Install dependencies

In [1]:
# Run once. Comment out lines you already have.
#!pip install -U edgartools sentence-transformers scikit-learn pandas -q

!pip install aspects: 
!import re
!import nltk

In [2]:
#!pip install nltk

#!pip install sentence-transformers
#!pip install scikit-learn
#!pip install pandas
#!pip install anthropic
#!pip install nltk
#!pip install numpy

In [3]:
#Set email and company name. 

In [4]:
# ── Config (interactive) ─────────────────────────────────────────

# ── Company name first ───────────────────────────────────────────
COMPANY_NAME = input("Company name (e.g., Microsoft, Apple, NVIDIA, Google): ").strip()

# ── Ticker ────────────────────────────────────────────────────────
TICKER = input(
    "Stock ticker (e.g., MSFT, AAPL, NVDA, GOOGL, META, AMZN, TSLA, CRM): "
).strip().upper()

# ── SEC email ─────────────────────────────────────────────────────
SEC_EMAIL = input("Your email (required by SEC EDGAR): ").strip()

# ── Fiscal year end month ─────────────────────────────────────────
fy_input = input(
    "Fiscal year end month (1-12). "
    "MSFT=6, AAPL=9, NVDA=1, CRM=1, ORCL=5, ADBE=11, COST=8. "
    "Most others (GOOGL, META, AMZN, TSLA, AMD, INTC, IBM) = 12. "
    "Press Enter for 12: "
).strip()
FISCAL_YEAR_END_MONTH = int(fy_input) if fy_input else 12

# ── Number of quarters ────────────────────────────────────────────
n_input = input("How many quarters of earnings? [default 5]: ").strip()
MAX_EARNINGS = int(n_input) if n_input else 5
MAX_SCAN = MAX_EARNINGS * 6   # scan ~6x as many 8-Ks (most aren't earnings)

# ── Chunking parameters — rarely need changing ───────────────────
MAX_TOKENS          = 400
OVERLAP_SENTENCES   = 1
MIN_CHUNK_CHARS     = 100
AVG_CHARS_PER_TOKEN = 4

# ── Confirmation ──────────────────────────────────────────────────
month_names = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
print("\n" + "="*50)
print(f"  Configured:")
print(f"    Company:       {COMPANY_NAME}")
print(f"    Ticker:        {TICKER}")
print(f"    SEC email:     {SEC_EMAIL}")
print(f"    Fiscal Y/E:    {month_names[FISCAL_YEAR_END_MONTH]} (month {FISCAL_YEAR_END_MONTH})")
print(f"    # Quarters:    {MAX_EARNINGS}")
print("="*50)


  Configured:
    Company:       Microsoft
    Ticker:        MSFT
    SEC email:     spso771522@gmail.com
    Fiscal Y/E:    Jun (month 6)
    # Quarters:    4


## 2. Configuration

Set the ticker, your contact email (SEC requires it), and how many quarters of earnings to pull.

In [5]:
import pandas as pd
from edgar import Company, set_identity

set_identity(SEC_EMAIL)

# ─── Extract text from a filing object ───────────────────────────
def extract_text_from_filing(filing_obj) -> str:
    """Try multiple methods to get text from a filing object."""

    # method 1: press release text (best for 8-K earnings)
    try:
        prs = filing_obj.press_releases
        if prs and len(prs) > 0:
            pr = prs[0]
            for attr in ["text", "markdown", "content", "html"]:
                if hasattr(pr, attr):
                    val = getattr(pr, attr)
                    if callable(val):
                        try:
                            result = val()
                            if isinstance(result, str) and len(result) > 200:
                                return result[:50000]
                        except Exception:
                            pass
                    elif isinstance(val, str) and len(val) > 200:
                        return val[:50000]
    except Exception:
        pass

    # method 2: filing .text() method
    try:
        text = filing_obj.text()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 3: filing .markdown() method
    try:
        text = filing_obj.markdown()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 4: str fallback
    try:
        text = str(filing_obj)
        if len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    return ""

# ─── Check if an 8-K is an earnings filing ───────────────────────
def is_earnings_filing(filing_obj) -> bool:
    """Item 2.02 = Results of Operations and Financial Condition."""
    try:
        items = filing_obj.items
        if any("2.02" in str(item) for item in items):
            return True
    except Exception:
        pass

    try:
        filing_str = str(filing_obj).lower()
        if any(kw in filing_str for kw in [
            "2.02", "results of operations",
            "quarterly results", "financial results",
            "earnings per share", "revenue was"
        ]):
            return True
    except Exception:
        pass

    return False

# ─── Pull N quarters of earnings 8-K ─────────────────────────────
def get_earnings_filings(ticker: str, company_name: str,
                          max_earnings: int = MAX_EARNINGS,
                          max_scan: int = MAX_SCAN) -> pd.DataFrame:
    print(f"\n{'='*60}")
    print(f"  EDGAR EARNINGS COLLECTION — {ticker}")
    print(f"  Target: last {max_earnings} earnings quarters")
    print(f"{'='*60}")

    company = Company(ticker)
    print(f"  ✓ Found: {company.name}")

    filings = company.get_filings(form="8-K").head(max_scan)
    results = []

    for filing in filings:
        try:
            filing_date = str(filing.filing_date)
            obj = filing.obj()

            if not is_earnings_filing(obj):
                continue

            text = extract_text_from_filing(obj)
            if not text or len(text.strip()) < 200:
                continue

            results.append({
                "filing_date": filing_date,
                "ticker":      ticker,
                "company":     company_name,
                "form":        "8-K",
                "text":        text,
            })
            print(f"  ✓ {filing_date}  ({len(text):,} chars)")

            if len(results) >= max_earnings:
                break
        except Exception as e:
            print(f"  ! skipped {filing.filing_date}: {e}")
            continue

    df = pd.DataFrame(results)
    print(f"\n  Collected {len(df)} earnings filings")
    return df

df_edgar = get_earnings_filings(TICKER, COMPANY_NAME)
df_edgar[['filing_date', 'ticker']]

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  EDGAR EARNINGS COLLECTION — MSFT
  Target: last 4 earnings quarters
  ✓ Found: MICROSOFT CORP
  ✓ 2026-04-29  (36,782 chars)
  ✓ 2026-01-28  (36,460 chars)
  ✓ 2025-10-29  (32,755 chars)
  ✓ 2025-07-30  (34,021 chars)

  Collected 4 earnings filings


,filing_date,ticker
0,2026-04-29,MSFT
1,2026-01-28,MSFT
2,2025-10-29,MSFT
3,2025-07-30,MSFT


## 3. Pull earnings 8-K filings from SEC EDGAR

In [6]:
import pandas as pd
from edgar import Company, set_identity

set_identity(SEC_EMAIL)

# ─── Extract text from a filing object ───────────────────────────
def extract_text_from_filing(filing_obj) -> str:
    """Try multiple methods to get text from a filing object."""

    # method 1: press release text (best for 8-K earnings)
    try:
        prs = filing_obj.press_releases
        if prs and len(prs) > 0:
            pr = prs[0]
            for attr in ["text", "markdown", "content", "html"]:
                if hasattr(pr, attr):
                    val = getattr(pr, attr)
                    if callable(val):
                        try:
                            result = val()
                            if isinstance(result, str) and len(result) > 200:
                                return result[:50000]
                        except Exception:
                            pass
                    elif isinstance(val, str) and len(val) > 200:
                        return val[:50000]
    except Exception:
        pass

    # method 2: filing .text() method
    try:
        text = filing_obj.text()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 3: filing .markdown() method
    try:
        text = filing_obj.markdown()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 4: str fallback
    try:
        text = str(filing_obj)
        if len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    return ""

# ─── Check if an 8-K is an earnings filing ───────────────────────
def is_earnings_filing(filing_obj) -> bool:
    """Item 2.02 = Results of Operations and Financial Condition."""
    try:
        items = filing_obj.items
        if any("2.02" in str(item) for item in items):
            return True
    except Exception:
        pass

    try:
        filing_str = str(filing_obj).lower()
        if any(kw in filing_str for kw in [
            "2.02", "results of operations",
            "quarterly results", "financial results",
            "earnings per share", "revenue was"
        ]):
            return True
    except Exception:
        pass

    return False

# ─── Pull N quarters of earnings 8-K ─────────────────────────────
def get_earnings_filings(ticker: str, company_name: str,
                          max_earnings: int = MAX_EARNINGS,
                          max_scan: int = MAX_SCAN) -> pd.DataFrame:
    print(f"\n{'='*60}")
    print(f"  EDGAR EARNINGS COLLECTION — {ticker}")
    print(f"  Target: last {max_earnings} earnings quarters")
    print(f"{'='*60}")

    company = Company(ticker)
    print(f"  ✓ Found: {company.name}")

    filings = company.get_filings(form="8-K").head(max_scan)
    results = []

    for filing in filings:
        try:
            filing_date = str(filing.filing_date)
            obj = filing.obj()

            if not is_earnings_filing(obj):
                continue

            text = extract_text_from_filing(obj)
            if not text or len(text.strip()) < 200:
                continue

            results.append({
                "filing_date": filing_date,
                "ticker":      ticker,
                "company":     company_name,
                "form":        "8-K",
                "text":        text,
            })
            print(f"  ✓ {filing_date}  ({len(text):,} chars)")

            if len(results) >= max_earnings:
                break
        except Exception as e:
            print(f"  ! skipped {filing.filing_date}: {e}")
            continue

    df = pd.DataFrame(results)
    print(f"\n  Collected {len(df)} earnings filings")
    return df

df_edgar = get_earnings_filings(TICKER, COMPANY_NAME)
df_edgar[['filing_date', 'ticker']]


  EDGAR EARNINGS COLLECTION — MSFT
  Target: last 4 earnings quarters
  ✓ Found: MICROSOFT CORP
  ✓ 2026-04-29  (36,782 chars)
  ✓ 2026-01-28  (36,460 chars)
  ✓ 2025-10-29  (32,755 chars)
  ✓ 2025-07-30  (34,021 chars)

  Collected 4 earnings filings


,filing_date,ticker
0,2026-04-29,MSFT
1,2026-01-28,MSFT
2,2025-10-29,MSFT
3,2025-07-30,MSFT


## 4. Derive quarter labels from filing dates

No hardcoded `QUARTER_MAP`. Uses fiscal-year-end month to produce labels like `FQ1-2026`.

In [7]:
from datetime import datetime

def filing_date_to_quarter(filing_date: str,
                            fy_end_month: int = FISCAL_YEAR_END_MONTH) -> str:
    """
    Approximate fiscal quarter label from filing date.
    A 10-Q/8-K earnings filing typically comes ~1 month after the
    fiscal quarter ends, so we back off one month before bucketing.
    """
    d = datetime.strptime(str(filing_date)[:10], "%Y-%m-%d")
    # shift back ~30 days so the filing date maps to the reporting quarter
    reporting_month = d.month - 1 if d.month > 1 else 12
    reporting_year  = d.year if d.month > 1 else d.year - 1

    # months since fiscal year end (modulo 12)
    months_into_fy = (reporting_month - fy_end_month - 1) % 12
    fq = months_into_fy // 3 + 1

    # fiscal year containing this reporting month
    if reporting_month > fy_end_month:
        fy = reporting_year + 1
    else:
        fy = reporting_year

    return f"FQ{fq}-{fy}"

# Apply to df_edgar for sanity check
df_edgar['quarter'] = df_edgar['filing_date'].apply(filing_date_to_quarter)

print("Quarter derivation check:")
print(df_edgar[['filing_date', 'quarter']].to_string(index=False))
print("\nIf the quarter labels look wrong, adjust FISCAL_YEAR_END_MONTH in Section 2.")

Quarter derivation check:
filing_date  quarter
 2026-04-29 FQ3-2026
 2026-01-28 FQ2-2026
 2025-10-29 FQ1-2026
 2025-07-30 FQ4-2025

If the quarter labels look wrong, adjust FISCAL_YEAR_END_MONTH in Section 2.


In [8]:
!pip install nltk

## 5. Clean and chunk the earnings text

In [17]:
import re
import nltk
from nltk.tokenize import sent_tokenize
from dataclasses import dataclass, field

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

# Strip patterns for earnings boilerplate
STRIP_PATTERNS_EARNINGS = [
    r"(?i)forward.looking statements.{0,2000}",
    r"(?i)safe harbor.{0,1000}",
    r"(?i)private securities litigation.{0,500}",
    r"(?i)(investor relations|media contact|for more information).{0,300}",
    r"(?i)^(document|exhibit\s*\d+\.\d+)\s*$",
]

@dataclass
class Chunk:
    text:         str
    company:      str
    ticker:       str
    source_type:  str
    filing_date:  str
    quarter:      str
    chunk_index:  int
    chunk_total:  int
    char_count:   int = field(init=False)
    token_count:  int = field(init=False)

    def __post_init__(self):
        self.char_count  = len(self.text)
        self.token_count = len(self.text) // AVG_CHARS_PER_TOKEN

def clean_text(text: str) -> str:
    if not text:
        return ""
    # strip HTML
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&[a-z]+;", " ", text)
    # strip legal boilerplate
    for pattern in STRIP_PATTERNS_EARNINGS:
        text = re.sub(pattern, "", text, flags=re.DOTALL | re.MULTILINE)
    # drop lines that are mostly numbers (financial table rows)
    lines, clean_lines = text.split('\n'), []
    for line in lines:
        stripped = line.strip()
        alpha    = sum(1 for c in stripped if c.isalpha())
        total    = len(stripped)
        if total == 0 or alpha / total >= 0.4:
            clean_lines.append(line)
    text = '\n'.join(clean_lines)
    # normalize whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

def semantic_chunk(text: str, max_tokens: int = MAX_TOKENS,
                    overlap: int = OVERLAP_SENTENCES) -> list:
    try:
        sentences = sent_tokenize(text)
    except Exception:
        sentences = re.split(r'(?<=[.!?])\s+', text)
    if not sentences:
        return []

    chunks, current, cur_tokens = [], [], 0
    for sent in sentences:
        sent_tokens = len(sent) // AVG_CHARS_PER_TOKEN
        if cur_tokens + sent_tokens > max_tokens and current:
            chunks.append(' '.join(current))
            current = current[-overlap:] if overlap > 0 else []
            cur_tokens = sum(len(s) // AVG_CHARS_PER_TOKEN for s in current)
        current.append(sent)
        cur_tokens += sent_tokens
    if current:
        chunks.append(' '.join(current))
    return [c for c in chunks if len(c) >= MIN_CHUNK_CHARS]

def chunk_earnings_report(raw_text: str, company: str, ticker: str,
                           filing_date: str, quarter: str) -> list:
    cleaned = clean_text(raw_text)
    if not cleaned:
        return []
    pieces = semantic_chunk(cleaned)
    total = len(pieces)
    return [
        Chunk(
            text=p, company=company, ticker=ticker,
            source_type='earnings', filing_date=filing_date,
            quarter=quarter, chunk_index=i, chunk_total=total,
        )
        for i, p in enumerate(pieces)
    ]

# ── Chunk all quarters ───────────────────────────────────────────
all_chunks = []
for _, row in df_edgar.iterrows():
    chunks = chunk_earnings_report(
        raw_text=row['text'], company=row['company'], ticker=row['ticker'],
        filing_date=row['filing_date'], quarter=row['quarter'],
    )
    all_chunks.extend(chunks)
    print(f"  ✓ {row['quarter']} ({row['filing_date']}): {len(chunks)} chunks")

df_earnings = pd.DataFrame([c.__dict__ for c in all_chunks])
print(f"\nTotal chunks: {len(df_earnings)}")
df_earnings.head(3)

  ✓ FQ3-2026 (2026-04-29): 10 chunks
  ✓ FQ2-2026 (2026-01-28): 10 chunks
  ✓ FQ1-2026 (2025-10-29): 9 chunks
  ✓ FQ4-2025 (2025-07-30): 8 chunks

Total chunks: 37


[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1002)>
[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1002)>


,text,company,ticker,source_type,filing_date,quarter,chunk_index,chunk_total,char_count,token_count
0,Microsoft Cloud and AI Strength Fuels Third Qu...,Microsoft,MSFT,earnings,2026-04-29,FQ3-2026,0,10,1543,385
1,The following table reconciles our financial r...,Microsoft,MSFT,earnings,2026-04-29,FQ3-2026,1,10,819,204
2,"Three Months Ended March 31, \n($ in millions,...",Microsoft,MSFT,earnings,2026-04-29,FQ3-2026,2,10,1883,470


## 6. Content novelty analysis (Cell A)

For each chunk in quarter N, novelty = `1 - max_cosine_similarity` to any chunk in quarter N-1.

In [18]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── 1. Embed every chunk ─────────────────────────────────────────
print("Loading embedder (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

df_earnings = df_earnings.reset_index(drop=True)

print(f"Embedding {len(df_earnings)} chunks...")
chunk_embeddings = embedder.encode(
    df_earnings['text'].astype(str).tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"✓ Embeddings shape: {chunk_embeddings.shape}")

# ── 2. Light boilerplate filter (TREC two-stage pattern) ─────────
BOILERPLATE_KEYWORDS = [
    'forward-looking', 'safe harbor', 'private securities litigation',
    'investor relations contact', 'media contact', 'cautionary statement',
    'this press release contains', 'non-gaap',
]

def is_boilerplate(text: str) -> bool:
    t = str(text).lower()
    if len(t) < 150:
        return True
    hits = sum(1 for kw in BOILERPLATE_KEYWORDS if kw in t)
    return hits >= 2

df_earnings['is_boilerplate'] = df_earnings['text'].apply(is_boilerplate)
n_boiler = df_earnings['is_boilerplate'].sum()
print(f"\nBoilerplate chunks filtered: {n_boiler} / {len(df_earnings)}")

# ── 3. Build chronological quarter order ─────────────────────────
quarter_order = (
    df_earnings[['quarter', 'filing_date']]
    .drop_duplicates().sort_values('filing_date').reset_index(drop=True)
)
print(f"\nQuarter order (oldest → newest):")
for _, row in quarter_order.iterrows():
    print(f"  {row['quarter']}  ({row['filing_date']})")

quarters_list = quarter_order['quarter'].tolist()

# ── 4. Compute novelty per chunk ─────────────────────────────────
novelty_scores    = np.full(len(df_earnings), np.nan)
nearest_prior_idx = np.full(len(df_earnings), -1, dtype=int)

for i in range(1, len(quarters_list)):
    cur_q, prev_q = quarters_list[i], quarters_list[i - 1]
    cur_idx  = df_earnings.index[df_earnings['quarter'] == cur_q].to_numpy()
    prev_idx = df_earnings.index[df_earnings['quarter'] == prev_q].to_numpy()

    if len(cur_idx) == 0 or len(prev_idx) == 0:
        print(f"  skip {prev_q} → {cur_q} (empty quarter)")
        continue

    sim     = cosine_similarity(chunk_embeddings[cur_idx], chunk_embeddings[prev_idx])
    max_sim = sim.max(axis=1)
    argmax  = sim.argmax(axis=1)

    novelty_scores[cur_idx]    = 1.0 - max_sim
    nearest_prior_idx[cur_idx] = prev_idx[argmax]

df_earnings['novelty']           = novelty_scores
df_earnings['nearest_prior_idx'] = nearest_prior_idx

Loading embedder (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5810.69it/s]


Embedding 37 chunks...


Batches: 100%|██████████| 2/2 [00:00<00:00, 11.18it/s]

✓ Embeddings shape: (37, 384)

Boilerplate chunks filtered: 3 / 37

Quarter order (oldest → newest):
  FQ4-2025  (2025-07-30)
  FQ1-2026  (2025-10-29)
  FQ2-2026  (2026-01-28)
  FQ3-2026  (2026-04-29)


## 7. Report top novel chunks per quarter

In [19]:
TOP_N = 5
print(f"{'='*72}")
print(f"  TOP {TOP_N} NOVEL CHUNKS PER QUARTER")
print(f"  (content most unlike anything in the previous quarter)")
print(f"{'='*72}")

for i in range(1, len(quarters_list)):
    cur_q, prev_q = quarters_list[i], quarters_list[i - 1]
    q_df = df_earnings[
        (df_earnings['quarter'] == cur_q)
        & (~df_earnings['is_boilerplate'])
        & (df_earnings['novelty'].notna())
    ].sort_values('novelty', ascending=False).head(TOP_N)

    print(f"\n── {cur_q} vs {prev_q} ──")
    if len(q_df) == 0:
        print("  (no substantive chunks with valid novelty)")
        continue
    for rank, (_, row) in enumerate(q_df.iterrows(), 1):
        prior = df_earnings.loc[row['nearest_prior_idx']]
        print(f"\n  [#{rank}] novelty = {row['novelty']:.3f}")
        print(f"  {cur_q}:   {str(row['text'])[:240].strip()}...")
        print(f"  closest {prev_q}: {str(prior['text'])[:200].strip()}...")

  TOP 5 NOVEL CHUNKS PER QUARTER
  (content most unlike anything in the previous quarter)

── FQ1-2026 vs FQ4-2025 ──

  [#1] novelty = 0.204
  FQ1-2026:   Quarterly Highlights, Product Releases, and Customer Stories
Every quarter Microsoft delivers hundreds of products, services, and enhancements. These releases are driven by years of significant research and development investments, to empow...
  closest FQ4-2025: Fiscal Year 2025 Results
Microsoft Corp. today announced the following results for the fiscal year ended June 30, 2025, as compared to the corresponding period of last fiscal year:
•Revenue was $281.7...

  [#2] novelty = 0.188
  FQ1-2026:   “It’s why we continue to increase our investments in AI across both capital and talent to meet the massive opportunity ahead.”
“We delivered a strong start to the fiscal year, exceeding expectations across revenue, operating income, and ear...
  closest FQ4-2025: Microsoft Cloud and AI Strength Fuels Fourth Quarter Results
REDMOND, Wash

In [12]:
# ── Latest vs prior quarter — uses results from Section 6 ────────
#TOP_N = 10

# quarters_list is already chronological (oldest → newest) from Section 6
#latest_q = quarters_list[-1]
#prior_q  = quarters_list[-2]

# Filter: latest quarter, substantive only, with valid novelty
#latest_novel = df_earnings[
#    (df_earnings['quarter'] == latest_q)
#    & (~df_earnings['is_boilerplate'])
#    & (df_earnings['novelty'].notna())
#].sort_values('novelty', ascending=False).head(TOP_N)

#print(f"{'='*72}")
#print(f"  WHAT'S NEW IN {latest_q} vs {prior_q}")
#print(f"  Top {TOP_N} most novel chunks")
#print(f"{'='*72}")

#for rank, (_, row) in enumerate(latest_novel.iterrows(), 1):
#    prior_chunk = df_earnings.loc[row['nearest_prior_idx']]
#    print(f"\n[#{rank}] novelty = {row['novelty']:.3f}")
#    print(f"\n{latest_q}:")
#    print(f"  {str(row['text'])[:400].strip()}{'...' if len(row['text']) > 400 else ''}")
#    print(f"\nClosest match in {prior_q}:")
#    print(f"  {str(prior_chunk['text'])[:300].strip()}{'...' if len(prior_chunk['text']) > 300 else ''}")
#    print('-' * 72)

# Quick summary
#print(f"\n{'='*72}")
#print(f"  SUMMARY: {latest_q} vs {prior_q}")
#print(f"{'='*72}")
#substantive = df_earnings[
#    (df_earnings['quarter'] == latest_q)
#    & (~df_earnings['is_boilerplate'])
#    & (df_earnings['novelty'].notna())
#]
#print(f"  Substantive chunks: {len(substantive)}")
#print(f"  Mean novelty:   {substantive['novelty'].mean():.3f}")
#print(f"  Median novelty: {substantive['novelty'].median():.3f}")
#print(f"  Max novelty:    {substantive['novelty'].max():.3f}")

## 8. Summary stats per quarter

In [20]:
summary = (
    df_earnings[~df_earnings['is_boilerplate']]
    .groupby('quarter')['novelty']
    .agg(['count', 'mean', 'median', 'max'])
    .rename(columns={'count': 'n_chunks'})
    .round(3)
)
summary = summary.reindex([q for q in quarters_list if q in summary.index])

print(f"{'='*60}")
print(f"  NOVELTY SUMMARY PER QUARTER")
print(f"{'='*60}")
print(summary.to_string())
print("\nInterpretation:")
print("  - The earliest quarter shows NaN (no prior to compare — correct).")
print("  - High mean novelty → broadly new material vs prior quarter.")
print("  - High max but moderate mean → one specific new topic introduced.")

  NOVELTY SUMMARY PER QUARTER
          n_chunks   mean  median    max
quarter                                 
FQ4-2025         0    NaN     NaN    NaN
FQ1-2026         8  0.116   0.103  0.204
FQ2-2026         9  0.038   0.039  0.073
FQ3-2026         9  0.030   0.013  0.125

Interpretation:
  - The earliest quarter shows NaN (no prior to compare — correct).
  - High mean novelty → broadly new material vs prior quarter.
  - High max but moderate mean → one specific new topic introduced.


In [21]:
# ── Install if needed ────────────────────────────────────────────
!pip install anthropic -q

In [23]:
# ── API Config (interactive) ─────────────────────────────────────
import os
import getpass

# Use getpass so the key doesn't show on screen as you type/paste
# Falls back to env var if it's already set, so you can avoid retyping
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key (sk-ant-...): ").strip()

assert ANTHROPIC_API_KEY.startswith("sk-ant-"), \
    "That doesn't look like an Anthropic key — should start with 'sk-ant-'"

# How many top novel chunks per transition to send to the LLM.
TOP_N_FOR_LLM = 10

# Model choice
MODEL = "claude-sonnet-4-5"

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")

✓ API key loaded
✓ Using model: claude-sonnet-4-5


In [28]:
# ── Build the quarter-transition pairs for the LLM ───────────────
# For each transition, assemble the top-N novel chunks along with
# their closest prior-quarter neighbor. The LLM will see the diff.
import pandas as pd

def build_transition_payload(df: pd.DataFrame,
                              cur_q: str, prev_q: str,
                              top_n: int = TOP_N_FOR_LLM) -> list:
    """Return a list of {novel_chunk, closest_prior_chunk, novelty} dicts."""
    q_df = df[
        (df['quarter'] == cur_q)
        & (~df['is_boilerplate'])
        & (df['novelty'].notna())
    ].sort_values('novelty', ascending=False).head(top_n)

    pairs = []
    for _, row in q_df.iterrows():
        prior = df.loc[row['nearest_prior_idx']]
        pairs.append({
            'novelty': round(float(row['novelty']), 3),
            'novel_chunk': str(row['text']).strip(),
            'closest_prior_chunk': str(prior['text']).strip(),
        })
    return pairs

# Verify it works by printing the FQ4-2025 payload shape
test_pairs = build_transition_payload(df_earnings, 'FQ4-2025', 'FQ3-2025')
# Use the latest quarter (which has novelty computed against the prior one)
test_pairs = build_transition_payload(df_earnings, 'FQ3-2026', 'FQ2-2026')

print(f"Built {len(test_pairs)} pairs for FQ2-2026 → FQ3-2026")
if test_pairs:
    print(f"First pair novelty: {test_pairs[0]['novelty']}")
    print(f"First novel chunk starts: {test_pairs[0]['novel_chunk'][:100]}...")
else:
    print("No pairs built — check that novelty was computed for this quarter.")

Built 9 pairs for FQ2-2026 → FQ3-2026
First pair novelty: 0.125
First novel chunk starts: The following table reconciles our financial results reported in accordance with generally accepted ...


In [26]:
  # ── The prompt ───────────────────────────────────────────────────
  # Prompt design notes:
  # - Structured JSON output with typed categories reduces editorializing
  # - Require verbatim quotes to ground every claim
  # - Explicit instruction NOT to comment on tone/hedging (that's Cell B)
  # - Neutral framing: "characterize what is new," not "interpret significance"

  SYSTEM_PROMPT = """You are a financial text analyst examining quarter-over-quarter changes in earnings press releases.

  You will receive pairs of text chunks. In each pair:
  - "novel_chunk" is a passage from the current quarter's earnings release
  - "closest_prior_chunk" is the most semantically similar passage from the previous quarter's release
  - "novelty" is a score from 0 to 1 where higher values mean less similar

  Your job: identify what NEW CONTENT appears in the current quarter that wasn't in the previous quarter. Classify each distinct new theme into one of these categories:

  - new_product: A product, service, SKU, or offering mentioned that wasn't in the prior quarter
  - new_metric: A financial or operating metric being reported that wasn't reported before, OR a new segment breakdown
  - new_risk: A risk factor, challenge, or headwind mentioned that wasn't in the prior quarter
  - new_segment: A business segment, customer type, or geography being discussed that wasn't before
  - new_framing: The same topic as last quarter but described in a substantively different way (e.g., new terminology, new emphasis, different framing of the same business)
  - other: New content that doesn't fit the above categories

  STRICT RULES:
  1. Every theme you identify MUST include a verbatim quote from the novel_chunk text. Put quotes in a "quote" field.
  2. Do NOT comment on tone, confidence, hedging, sentiment, or how cautious/bullish the language sounds. Only describe what content is new.
  3. Do NOT speculate about why the change happened or what it means for the business. Only describe what is different.
  4. If a "novel_chunk" is actually similar to its prior chunk (just reworded), say so in the "notes" field and mark it as new_framing with low confidence.
  5. Consolidate themes across multiple chunks — if three chunks all introduce the same new product, output ONE theme with all three quotes.

  Output format: valid JSON only, no other text. Schema:
  {
    "themes": [
      {
        "category": "new_product" | "new_metric" | "new_risk" | "new_segment" | "new_framing" | "other",
        "label": "Short descriptive label (3-8 words)",
        "description": "1-2 sentence neutral description of what's new",
        "quotes": ["verbatim phrase from novel_chunk", "another verbatim phrase"],
        "confidence": "high" | "medium" | "low",
        "notes": "any caveats about whether this is genuinely new"
      }
    ]
  }"""

  def build_user_message(prev_q: str, cur_q: str, pairs: list) -> str:
      """Format the payload as the user message."""
      lines = [
          f"Transition: {prev_q} → {cur_q}",
          f"Number of pairs: {len(pairs)}",
          "",
          "Pairs (sorted by novelty, highest first):",
          "",
      ]
      for i, p in enumerate(pairs, 1):
          lines.append(f"--- Pair {i} (novelty={p['novelty']}) ---")
          lines.append(f"NOVEL [{cur_q}]:")
          lines.append(p['novel_chunk'])
          lines.append("")
          lines.append(f"CLOSEST PRIOR [{prev_q}]:")
          lines.append(p['closest_prior_chunk'])
          lines.append("")
      return "\n".join(lines)

  # Preview what gets sent for one transition
  preview = build_user_message('FQ3-2025', 'FQ4-2025', test_pairs[:2])
  print(preview[:1500])
  print("...(truncated for preview)")


Transition: FQ3-2025 → FQ4-2025
Number of pairs: 2

Pairs (sorted by novelty, highest first):

--- Pair 1 (novelty=0.125) ---
NOVEL [FQ4-2025]:
The following table reconciles our financial results reported in accordance with generally accepted accounting principles (GAAP) to non-GAAP financial results. Additional information regarding our non-GAAP definition is provided below. All growth comparisons relate to the corresponding period in the last fiscal year. Three Months Ended March 31, 
($ in millions, except per share amounts) As Reported (GAAP) Adjustment* As Adjusted (non-GAAP) As Reported (GAAP) Adjustment* As Adjusted (non-GAAP) GAAP Constant Currency Non-GAAP Non-GAAP Constant Currency
Income 
per Share 
*Adjustment is the impact from investments in OpenAI
Business Highlights
Microsoft Cloud revenue was $54.5 billion and increased 29% (up 25% in constant currency), and commercial remaining performance obligation increased 99% to $627 billion.

CLOSEST PRIOR [FQ3-2025]:
“We are p

In [29]:
# ── Build quarter × quarter novelty matrix ───────────────────────
# nov_matrix.loc[qi, qj] = average novelty of qi's non-boilerplate chunks
# vs. their closest matches in qj. Diagonal is 0 by definition.
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

nov_matrix = pd.DataFrame(
    index=quarters_list, columns=quarters_list, dtype=float
)

for qi in quarters_list:
    idx_i = df_earnings[
        (df_earnings['quarter'] == qi) & (~df_earnings['is_boilerplate'])
    ].index
    if len(idx_i) == 0:
        nov_matrix.loc[qi, :] = np.nan
        continue

    emb_i = chunk_embeddings[idx_i]

    for qj in quarters_list:
        if qi == qj:
            nov_matrix.loc[qi, qj] = 0.0
            continue

        idx_j = df_earnings[
            (df_earnings['quarter'] == qj) & (~df_earnings['is_boilerplate'])
        ].index
        if len(idx_j) == 0:
            nov_matrix.loc[qi, qj] = np.nan
            continue

        emb_j = chunk_embeddings[idx_j]
        sims = cosine_similarity(emb_i, emb_j)
        nov_matrix.loc[qi, qj] = float((1 - sims.max(axis=1)).mean())

print("Quarter × Quarter Novelty Matrix")
print("(rows = source quarter, cols = compared-against quarter)")
print(nov_matrix.round(3))

Quarter × Quarter Novelty Matrix
(rows = source quarter, cols = compared-against quarter)
          FQ4-2025  FQ1-2026  FQ2-2026  FQ3-2026
FQ4-2025     0.000     0.101     0.100     0.103
FQ1-2026     0.116     0.000     0.035     0.055
FQ2-2026     0.113     0.038     0.000     0.030
FQ3-2026     0.119     0.054     0.030     0.000


In [30]:
# ── Timeline: latest quarter vs. every prior quarter ─────────────
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

latest_q = quarters_list[-1]
prior_quarters = quarters_list[:-1]  # chronological: oldest → newest

# Build the timeline table
timeline_rows = []
for q in prior_quarters:
    score = nov_matrix.loc[latest_q, q]
    months_back = (len(quarters_list) - 1) - quarters_list.index(q)  # quarters back
    timeline_rows.append({
        'prior_quarter': q,
        'quarters_back': months_back,
        'novelty_vs_latest': round(float(score), 3),
    })

timeline = pd.DataFrame(timeline_rows)

print("="*72)
print(f"  TIMELINE: {latest_q} compared against every prior quarter")
print(f"  (chronological order — oldest first)")
print("="*72)
print(timeline.to_string(index=False))

# Identify extremes
most_diff = timeline.loc[timeline['novelty_vs_latest'].idxmax()]
most_sim  = timeline.loc[timeline['novelty_vs_latest'].idxmin()]

print(f"\n  ▲ MOST DIFFERENT from {latest_q}:")
print(f"     {most_diff['prior_quarter']}  "
      f"(novelty = {most_diff['novelty_vs_latest']:.3f}, "
      f"{int(most_diff['quarters_back'])} quarters back)")

print(f"\n  ▼ MOST SIMILAR to {latest_q}:")
print(f"     {most_sim['prior_quarter']}  "
      f"(novelty = {most_sim['novelty_vs_latest']:.3f}, "
      f"{int(most_sim['quarters_back'])} quarters back)")

# ── ASCII bar chart for quick visual ─────────────────────────────
print(f"\n{'='*72}")
print(f"  VISUAL TIMELINE")
print(f"{'='*72}")
max_score = timeline['novelty_vs_latest'].max()
bar_width = 40
for _, row in timeline.iterrows():
    bar_len = int((row['novelty_vs_latest'] / max_score) * bar_width) if max_score > 0 else 0
    bar = '█' * bar_len
    marker = ''
    if row['prior_quarter'] == most_diff['prior_quarter']:
        marker = '  ← most different'
    elif row['prior_quarter'] == most_sim['prior_quarter']:
        marker = '  ← most similar'
    print(f"  {row['prior_quarter']:>10}  {bar:<{bar_width}}  {row['novelty_vs_latest']:.3f}{marker}")

# ── Show what's driving the biggest gap ──────────────────────────
print(f"\n{'='*72}")
print(f"  TOP 5 PASSAGES IN {latest_q} MOST UNLIKE ANYTHING IN {most_diff['prior_quarter']}")
print(f"{'='*72}")

idx_latest = df_earnings[(df_earnings['quarter'] == latest_q)
                         & (~df_earnings['is_boilerplate'])].index
idx_far = df_earnings[(df_earnings['quarter'] == most_diff['prior_quarter'])
                      & (~df_earnings['is_boilerplate'])].index

sims = cosine_similarity(chunk_embeddings[idx_latest], chunk_embeddings[idx_far])
novelty_per_chunk = 1 - sims.max(axis=1)

ranked = pd.DataFrame({
    'novelty': novelty_per_chunk,
    'text': df_earnings.loc[idx_latest, 'text'].values,
}).sort_values('novelty', ascending=False).head(5)

for rank, (_, row) in enumerate(ranked.iterrows(), 1):
    text = row['text'][:280]
    print(f"\n  [#{rank}] novelty = {row['novelty']:.3f}")
    print(f"      {text}{'...' if len(row['text']) > 280 else ''}")

  TIMELINE: FQ3-2026 compared against every prior quarter
  (chronological order — oldest first)
prior_quarter  quarters_back  novelty_vs_latest
     FQ4-2025              3              0.119
     FQ1-2026              2              0.054
     FQ2-2026              1              0.030

  ▲ MOST DIFFERENT from FQ3-2026:
     FQ4-2025  (novelty = 0.119, 3 quarters back)

  ▼ MOST SIMILAR to FQ3-2026:
     FQ2-2026  (novelty = 0.030, 1 quarters back)

  VISUAL TIMELINE
    FQ4-2025  ████████████████████████████████████████  0.119  ← most different
    FQ1-2026  ██████████████████                        0.054
    FQ2-2026  ██████████                                0.030  ← most similar

  TOP 5 PASSAGES IN FQ3-2026 MOST UNLIKE ANYTHING IN FQ4-2025

  [#1] novelty = 0.272
      Quarterly Highlights, Product Releases, and Customer Stories
Every quarter Microsoft delivers hundreds of products, services, and enhancements. These releases are driven by years of significant research and develo

In [31]:
# ── Call Claude and parse the response ───────────────────────────
import anthropic
import json

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def summarize_transition(prev_q: str, cur_q: str, pairs: list) -> dict:
    if len(pairs) == 0:
        return {"themes": [], "error": "no pairs available"}

    user_msg = build_user_message(prev_q, cur_q, pairs)

    response = client.messages.create(
        model=MODEL,
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}],
    )

    raw = response.content[0].text.strip()

    # Strip markdown code fences if the model wrapped its JSON
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as e:
        return {
            "themes": [],
            "error": f"JSON parse failed: {e}",
            "raw_response": raw[:500],
        }

    # Attach usage stats for cost tracking
    parsed['_meta'] = {
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
        'pairs_sent': len(pairs),
    }
    return parsed

In [32]:
# ── Compare LATEST quarter against every prior quarter ───────────
# Uses `quarters_list` from Section 6 (already defined in your notebook).

transition_summaries = {}

cur_q = quarters_list[-1]            # the latest quarter
prior_quarters = quarters_list[:-1]  # everything before it

print(f"Comparing latest quarter {cur_q} against {len(prior_quarters)} prior quarters\n")

for prev_q in prior_quarters:
    print(f"\n{'='*60}")
    print(f"  Analyzing {prev_q} → {cur_q}")
    print(f"{'='*60}")

    pairs = build_transition_payload(df_earnings, cur_q, prev_q)
    if len(pairs) == 0:
        print(f"  (no valid pairs — skipping)")
        continue

    print(f"  Sending {len(pairs)} pairs to Claude...")
    result = summarize_transition(prev_q, cur_q, pairs)
    transition_summaries[f"{prev_q}->{cur_q}"] = result

    if 'error' in result:
        print(f"  ! error: {result['error']}")
        continue

    meta = result.get('_meta', {})
    print(f"  ✓ got {len(result['themes'])} themes"
          f"  ({meta.get('input_tokens', '?')} in / {meta.get('output_tokens', '?')} out tokens)")

Comparing latest quarter FQ3-2026 against 3 prior quarters


  Analyzing FQ4-2025 → FQ3-2026
  Sending 9 pairs to Claude...
  ✓ got 7 themes  (7076 in / 1138 out tokens)

  Analyzing FQ1-2026 → FQ3-2026
  Sending 9 pairs to Claude...
  ✓ got 7 themes  (7076 in / 1130 out tokens)

  Analyzing FQ2-2026 → FQ3-2026
  Sending 9 pairs to Claude...
  ✓ got 7 themes  (7076 in / 1099 out tokens)


In [33]:
# ── Pretty-print the results ─────────────────────────────────────

CATEGORY_ORDER = ['new_product', 'new_segment', 'new_metric',
                  'new_risk', 'new_framing', 'other']

def print_transition(key: str, summary: dict):
    print(f"\n{'═'*72}")
    print(f"  {key}")
    print(f"{'═'*72}")

    if 'error' in summary:
        print(f"  ERROR: {summary['error']}")
        if 'raw_response' in summary:
            print(f"  Raw: {summary['raw_response']}")
        return

    themes = summary.get('themes', [])
    if not themes:
        print("  (no themes identified)")
        return

    # Group by category in consistent order
    by_cat = {}
    for t in themes:
        by_cat.setdefault(t.get('category', 'other'), []).append(t)

    for cat in CATEGORY_ORDER:
        if cat not in by_cat:
            continue
        print(f"\n  {cat.upper()}:")
        for t in by_cat[cat]:
            conf = t.get('confidence', '?')
            label = t.get('label', '(no label)')
            desc = t.get('description', '')
            print(f"    • [{conf}] {label}")
            if desc:
                # wrap at 70 chars, indent continuation lines
                import textwrap
                wrapped = textwrap.fill(desc, width=70,
                                         initial_indent='      ',
                                         subsequent_indent='      ')
                print(wrapped)
            for q in t.get('quotes', [])[:2]:  # show up to 2 quotes
                print(f'        └ "{q[:150]}{"..." if len(q) > 150 else ""}"')
            notes = t.get('notes', '')
            if notes:
                print(f"      note: {notes}")

for key, summary in transition_summaries.items():
    print_transition(key, summary)

    #final output. 


════════════════════════════════════════════════════════════════════════
  FQ4-2025->FQ3-2026
════════════════════════════════════════════════════════════════════════

  NEW_METRIC:
    • [high] AI business annual revenue run rate disclosed
      Microsoft now reports a specific annual revenue run rate for its
      AI business ($37 billion, up 123% year-over-year), which was not
      disclosed in the prior quarter.
        └ "Our AI business surpassed an annual revenue run rate of $37 billion, up 123% year-over-year"
      note: The prior quarter mentioned that 'Microsoft has built an AI business that is larger than some of our biggest franchises' but did not provide specific revenue run rate figures.
    • [high] Commercial remaining performance obligation decreased quarter-over-quarter
      The commercial remaining performance obligation decreased from
      $625 billion (prior quarter) to $627 billion, but the growth
      rate changed from 110% to 99%, indicating a deceleration

In [34]:
# ── Output that is interesting: Print only the most recent quarter transition ──────────────── 

def print_latest_transition(transition_summaries: dict, quarters_list: list):
    """
    Print only the diff between the most recent quarter and the one before it.
    Reuses the existing print_transition() formatter.
    """
    if len(quarters_list) < 2:
        print("  Need at least 2 quarters to show a transition.")
        return
    
    # quarters_list is already chronological (oldest → newest)
    latest_q = quarters_list[-1]
    prior_q  = quarters_list[-2]
    key = f"{prior_q}->{latest_q}"
    
    if key not in transition_summaries:
        print(f"  No summary found for {key}")
        print(f"  Available transitions: {list(transition_summaries.keys())}")
        return
    
    print_transition(key, transition_summaries[key])


# Call it
print_latest_transition(transition_summaries, quarters_list)


════════════════════════════════════════════════════════════════════════
  FQ2-2026->FQ3-2026
════════════════════════════════════════════════════════════════════════

  NEW_METRIC:
    • [high] AI business annual revenue run rate
      Microsoft now reports an annual revenue run rate for its AI
      business, stating it surpassed $37 billion and grew 123% year-
      over-year. This specific metric was not disclosed in the prior
      quarter.
        └ "Our AI business surpassed an annual revenue run rate of $37 billion, up 123% year-over-year."
      note: This is a new quantified metric. The prior quarter mentioned AI business size qualitatively ('larger than some of our biggest franchises') but did not provide a specific dollar figure or growth rate.
    • [high] Microsoft Cloud revenue absolute value
      Microsoft Cloud revenue is now reported as $54.5 billion (up
      from the prior quarter's $51.5 billion crossing the $50 billion
      milestone). The company also now repo

In [37]:
# ── Flatten to a dataframe for downstream use ────────────────────
# Useful if you want to filter, export, or chart across quarters.

rows = []
for key, summary in transition_summaries.items():
    if 'error' in summary:
        continue
    prev_q, cur_q = key.split('->')
    for t in summary.get('themes', []):
        rows.append({
            'prev_quarter': prev_q,
            'cur_quarter': cur_q,
            'category': t.get('category'),
            'label': t.get('label'),
            'description': t.get('description'),
            'confidence': t.get('confidence'),
            'n_quotes': len(t.get('quotes', [])),
            'notes': t.get('notes', ''),
        })

df_themes = pd.DataFrame(rows)
print(f"Total themes across all transitions: {len(df_themes)}")
print("\nBy category:")
print(df_themes['category'].value_counts())
print("\nBy transition:")
print(df_themes.groupby('cur_quarter').size())

df_themes.head(20)

Total themes across all transitions: 21

By category:
category
new_metric     14
new_framing     4
other           3
Name: count, dtype: int64

By transition:
cur_quarter
FQ3-2026    21
dtype: int64


,prev_quarter,cur_quarter,category,label,description,confidence,n_quotes,notes
0,FQ4-2025,FQ3-2026,new_framing,Agentic computing era terminology introduced,Microsoft now frames its AI strategy around 't...,high,1,This represents new framing and terminology fo...
1,FQ4-2025,FQ3-2026,new_metric,AI business annual revenue run rate disclosed,Microsoft now reports a specific annual revenu...,high,1,The prior quarter mentioned that 'Microsoft ha...
2,FQ4-2025,FQ3-2026,new_metric,Commercial remaining performance obligation de...,The commercial remaining performance obligatio...,high,1,"While this metric was reported last quarter, t..."
3,FQ4-2025,FQ3-2026,new_metric,Microsoft Cloud revenue crossing $54.5 billion...,Microsoft Cloud revenue reached $54.5 billion ...,medium,1,The prior quarter emphasized 'Microsoft Cloud ...
4,FQ4-2025,FQ3-2026,new_metric,Search advertising vs search and news advertis...,The search advertising metric is now labeled '...,medium,1,Prior quarter used 'Search and news advertisin...
5,FQ4-2025,FQ3-2026,other,Brian DeFoe role change to deputy general counsel,"The webcast details now list 'Brian DeFoe, dep...",high,1,This appears to be a personnel change in who p...
6,FQ4-2025,FQ3-2026,new_metric,OpenAI investment impact reduced substantially,The net losses from OpenAI investments decreas...,high,1,Prior quarter showed a net gain of $7.6 billio...
7,FQ1-2026,FQ3-2026,new_metric,AI Business Annual Revenue Run Rate,Microsoft now reports an annual revenue run ra...,high,1,This is a new quantitative metric that provide...
8,FQ1-2026,FQ3-2026,new_framing,Agentic Computing Era Terminology,The CEO's statement now frames Microsoft's mis...,high,1,This represents new terminology to describe th...
9,FQ1-2026,FQ3-2026,new_metric,Microsoft Cloud Revenue Crosses $54 Billion,Microsoft Cloud revenue is now reported at $54...,medium,1,While this is the same metric category as last...


In [36]:
df_themes # aggreagted result in case for future use. 

,prev_quarter,cur_quarter,category,label,description,confidence,n_quotes,notes
0,FQ4-2025,FQ3-2026,new_framing,Agentic computing era terminology introduced,Microsoft now frames its AI strategy around 't...,high,1,This represents new framing and terminology fo...
1,FQ4-2025,FQ3-2026,new_metric,AI business annual revenue run rate disclosed,Microsoft now reports a specific annual revenu...,high,1,The prior quarter mentioned that 'Microsoft ha...
2,FQ4-2025,FQ3-2026,new_metric,Commercial remaining performance obligation de...,The commercial remaining performance obligatio...,high,1,"While this metric was reported last quarter, t..."
3,FQ4-2025,FQ3-2026,new_metric,Microsoft Cloud revenue crossing $54.5 billion...,Microsoft Cloud revenue reached $54.5 billion ...,medium,1,The prior quarter emphasized 'Microsoft Cloud ...
4,FQ4-2025,FQ3-2026,new_metric,Search advertising vs search and news advertis...,The search advertising metric is now labeled '...,medium,1,Prior quarter used 'Search and news advertisin...
5,FQ4-2025,FQ3-2026,other,Brian DeFoe role change to deputy general counsel,"The webcast details now list 'Brian DeFoe, dep...",high,1,This appears to be a personnel change in who p...
6,FQ4-2025,FQ3-2026,new_metric,OpenAI investment impact reduced substantially,The net losses from OpenAI investments decreas...,high,1,Prior quarter showed a net gain of $7.6 billio...
7,FQ1-2026,FQ3-2026,new_metric,AI Business Annual Revenue Run Rate,Microsoft now reports an annual revenue run ra...,high,1,This is a new quantitative metric that provide...
8,FQ1-2026,FQ3-2026,new_framing,Agentic Computing Era Terminology,The CEO's statement now frames Microsoft's mis...,high,1,This represents new terminology to describe th...
9,FQ1-2026,FQ3-2026,new_metric,Microsoft Cloud Revenue Crosses $54 Billion,Microsoft Cloud revenue is now reported at $54...,medium,1,While this is the same metric category as last...


Edited, Final saved on May 2nd, 2026. Adding additional implementation for the call information:
Recently, there was an earning call for Apple as CEO is transitioning  to executive chairman in 2026. So the below code compares recent April 30th Q2 earning call to Q1 earning call.

In [38]:
#Testing for earning calls.
text = """
Apple Inc. (AAPL) Q1 2026 Earnings Call Transcript
Jan 29, 2026, 9:02 PM ETApple Inc. (AAPL) Stock, AAPL:CA Stock, ZAAP:CA Stock
SA Transcripts
159.61K Followers
•	Save
•	
Comments

Q1: 2026-01-29 Earnings Summary
EPS of $2.84 beats by $0.17
 | Revenue of $143.76B (15.65% Y/Y) beats by $5.23B
Apple Inc. (AAPL) Q1 2026 Earnings Call January 29, 2026 5:00 PM EST
Company Participants
Suhasini Chandramouli - Director of Investor Relations
Timothy Cook - CEO & Director
Kevan Parekh - Senior VP & CFO
Conference Call Participants
Amit Daryanani - Evercore ISI Institutional Equities, Research Division
Erik Woodring - Morgan Stanley, Research Division
Michael Ng - Goldman Sachs Group, Inc., Research Division
Benjamin Reitzes - Melius Research LLC
David Vogt - UBS Investment Bank, Research Division
Wamsi Mohan - BofA Securities, Research Division
Samik Chatterjee - JPMorgan Chase & Co, Research Division
Sreekrishnan Sankarnarayanan - TD Cowen, Research Division
Atif Malik - Citigroup Inc., Research Division
Aaron Rakers - Wells Fargo Securities, LLC, Research Division
Richard Kramer - Arete Research Services LLP
Presentation
Suhasini Chandramouli
Director of Investor Relations
Good afternoon, and welcome to the Apple Q1 Fiscal Year 2026 Earnings Conference Call. My name is Suhasini Chandramouli, Director of Investor Relations. Today's call is being recorded. Speaking first today is Apple's CEO, Tim Cook, and he'll be followed by CFO, Kevan Parekh. After that, we'll open the call to questions from analysts.
Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including, without limitation, those regarding revenue, gross margin, operating expenses, other income and expense, taxes, capital allocation and future business outlook. These statements involve risks and uncertainties that may cause actual results or trends to differ materially from our forecast including risks related to the potential impact to the company's business and results of operations from macroeconomic conditions, tariffs and other measures and legal and regulatory proceedings.
For more information, please refer to the risk factors discussed in Apple's most recently filed reports on Form 10-Q and Form 10-K and the Form 8-K filed with the SEC today, along with the associated press release. Additional information will also be in our report on Form 10-Q for the quarter ended December 27, 2025, to be filed tomorrow and in other reports and filings we make with the SEC. Apple assumes no obligation to update any forward-looking statements, which speak only as of the date they are made.
I'd now like to turn the call over to Tim for introductory remarks.
Timothy Cook
CEO & Director
Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. I am proud to say that we just had a quarter for the record books. We are reporting our best-ever quarter with $143.8 billion in revenue, up 16% from a year ago and exceeding our expectations. The demand for iPhone was simply staggering with revenue growing 23% year-over-year and all-time records across every geographic segment.
Services set an all-time revenue record as well, up 14% from a year ago, and EPS reached an all-time record of $2.84, growing a robust 19% year-over-year. We set all-time revenue records in the Americas, Europe, Japan and rest of Asia Pacific and grew in the vast majority of markets we track. We continue to gain momentum in emerging markets which includes India, where we saw strong double-digit revenue growth. Greater China also grew 38% year-over-year, driven by iPhone, which had record upgraders and double-digit growth on switchers.
Apple's December quarter results underscore our relentless commitment to innovation to our customers and to our mission to build the best products and services in the world.
Now I'd like to take a closer look at results from across our lineup, beginning with iPhone. As I mentioned earlier, it was a fantastic quarter for iPhone with an all-time revenue record of $85.3 billion, up 23% year-over-year. This is the strongest iPhone lineup we've ever had and by far, the most popular. Throughout the quarter, customer enthusiasm for iPhone was simply extraordinary. Users were incredibly excited about everything enables them to do. iPhone 17 Pro and 17 Pro Max delivered the ultimate iPhone experience. They feature the best ever performance in battery life on an iPhone, the most advanced camera system and a striking design. iPhone Air, our slimmest and lightest smartphone yet, packs powerful capabilities into an ultra slim and sleek design. And iPhone 17 is a truly fantastic upgrade at an incredible value.
Turning to Mac. Revenue was $8.4 billion for the December quarter. We were pleased to see the Mac installed base reach another all-time high with nearly half of customers who purchased the Mac being new to the product. The M5 powered 14-inch MacBook Pro takes a huge leap in AI performance, thanks to the next-generation GPU architecture and a faster neural engine.
From the world's most popular laptop for consumers and businesses in MacBook Air to the small and spectacular Mac mini, every Mac in our lineup has something special to offer users. And with the recently released Apple Creator Studio available across Mac, iPad and iPhone, creators have more tools at their fingertips to make incredible music or turn their devices into a video production studio.
Meanwhile, iPad saw December quarter revenue of $8.6 billion, up 6% from a year ago with an all-time record for upgraders. We are proud to have our strongest lineup ever from iPad powered by A16, which is proving to be incredibly popular to iPad Air with its amazing versatility to the unbelievably powerful M5 iPad Pro with its remarkably thin and light design. It's no wonder that iPad continued to be the most popular tablet in the world. Across Wearables, Home and Accessories, revenue was $11.5 billion. With Apple Watch Ultra 3 and Apple Watch Series 11, users are tapping into a comprehensive set of health and wellness features to help them meet their health goals.
In a recent survey, we see an increasing number of users telling us they're wearing their watch to sleep, which allows them to check their sleep scores each morning and find ways to improve their sleep quality. And Apple Watch alerts are enabling important conversations between users and their doctors regarding potential signs of hypertension.
These are just some of the many ways that watch is helping people live healthier lives. The response to AirPods Pro 3 has been amazing. Customers are raving about the rich immersive sound quality, the unmatched level of active noise cancellation and the noticeably improved comfort that makes them effortless to wear.
Features like live translation are also changing the way people can communicate by helping users connect across languages in real time and making everyday conversations feel more natural and accessible. Together, these innovations create an experience that feels both powerful and personal and the enthusiasm we are seeing reflects just how strongly AirPods Pro 3 are resonating with customers.
Across our product categories, we are seeing very high levels of customer satisfaction, and we are proud to report that we have a new record for our installed base with more than 2.5 billion active devices. During the quarter, we were excited to see that the majority of users on enabled iPhones are actively leveraging the power of Apple Intelligence. Since the launch of Apple Intelligence, we've introduced dozens of features, including writing tools and cleanup and made it available in 15 languages.
These AI experiences are personal, private, integrated across our platforms and relevant to what our users do every day. We are bringing intelligence to more of what people already love about our products so we can make every experience even more capable and effortless.
One of our most popular features is visual intelligence which helps users learn and do more than ever with the content on their iPhone screen, making it faster to search, take action and answer questions across their apps. And as I touched on earlier, we are hearing powerful stories of people using live translation to communicate seamlessly across languages.
And these are just some of the many powerful AI features that are enabling our users to do remarkable things with our products, which are far and away the best platforms in the world for AI. That's a no small part because of the extraordinary power and performance of Apple silicon. Building on our efforts in the AI space, we are also collaborating with Google to develop the next generation of Apple Foundation Models. This will help power future Apple Intelligence features, including a more personalized Siri coming this year. We're incredibly excited for what's to come with so many new experiences to unlock.
Turning to services. We achieved an all-time revenue record of $30 billion, 14% higher from a year ago. Services also set all-time revenue records in both developed and emerging markets. Apple TV has seen fantastic momentum with December seeing a 36% increase in viewership over the previous year. It's no wonder with shows like Pluribus, which are creating landmark cultural moments that audiences are loving. Anticipation is building for upcoming new productions like Cape Fear from Steven Spielberg and Martin Scorsese. And we are thrilled to announce that Ted Lasso will be returning for a fourth season this summer.
Six years since launch, we're excited by the growing enthusiasm viewers have for Apple TV, and we are grateful for the accolades that have followed, most recently at the Critics Choice and Golden Globe Awards. To date, Apple TV productions have earned more than 650 wins and more than 3,200 nominations, including a recently announced Oscar nomination for Best Picture for F1, the movie. And speaking of F1, we're also approaching the start of a new Formula 1 season, and for F1 fans in the U.S., Apple TV will be the place to watch every practice, qualifying, sprint and grand prix. MLS fans will also be able to watch every regular and post-season game with their Apple TV subscription this year, and we're looking forward to kick off in the coming weeks.
Looking back, 2025 was a fantastic year for services as we rolled out amazing new features and broke records. Apple Music climbed to all-time highs in both listenership and new subscriber growth. Apple Pay eliminated more than $1 billion in fraud for our partners last year, and we've made it available in more markets than ever before. And last year, we welcomed more than 850 million users every week on average to the App Store, the world's safest and most innovative app marketplace. Developers have now earned more than $550 billion on our platform since 2008.
In retail, we continue to bring a magical experience to our customers all around the world, and we were thrilled to have our best ever results in retail during the quarter. We were excited to open our fifth store in India in December and have plans to open another store in Mumbai soon.
Wherever we are, we see ourselves as part of a larger whole. That's why we show up with our values in everything we do. That means working with partners in places like Vietnam to bring more clean water to rural areas. It means celebrating graduations of new classes of innovators from our developer academies in places such as Brazil, Indonesia and South Korea. It means 3D printing titanium cases for Apple Watch using recycled materials so that they're better for the planet without compromising quality and so much more.
We're especially proud of the work we're doing to support American innovation. Last year, we committed to invest $600 billion over 4 years in vital industries like advanced manufacturing, silicon engineering and artificial intelligence. As we're building on our long-standing investments in America, we're supporting nearly 0.5 million jobs with thousands of suppliers across all 50 states.
In the year since we made our initial commitment, we're making great progress. Today, we're shipping servers to power Apple Intelligence from our new manufacturing facility in Houston. Through our advanced manufacturing program, we're working with Corning in Kentucky to make 100% of cover glass for iPhone and Apple Watch. We're working with Micron, which broke ground on a new advanced chip packaging and test facility, and we continue to advance the development of an end-to-end silicon supply chain across the country sourcing 20 billion U.S. chips in 2025.
Through our Apple Manufacturing Academy in Detroit, we're already training American businesses and innovators on the latest smart manufacturing and artificial intelligence techniques. Six months since opening, the Academy is already making an enormously positive impact for businesses working alongside Apple engineers to drive productivity, efficiency and quality in their supply chains. As I said at the beginning of my remarks, this was in so many ways, a remarkable quarter for Apple. And we're excited for all the opportunities we'll have in the year ahead to deliver innovations that have never been seen before and enrich the lives of users every step of the way.
With so much to look forward to in the weeks and months ahead, I have every confidence that our best work is yet to come.
With that, I'll turn it over to Kevan.
Kevan Parekh
Senior VP & CFO
Thanks, Tim, and good afternoon, everyone. Our revenue of $143.8 billion was up 16% year-over-year, our best quarter ever. Across the world, we set all-time revenue records in both developed and emerging markets. And we saw double-digit growth year-over-year across the majority of the markets we track, including the U.S., Latin America, Western Europe, Greater China, India and South Asia.
Products revenue was $113.7 billion, up 16% year-over-year, driven by double-digit growth in iPhone, setting a new all-time record. And as Tim mentioned, thanks to our strong levels of customer loyalty and satisfaction, our installed base of active devices has now surpassed 2.5 billion, reaching another all-time high across all product categories and geographic segments.
Services revenue was $30 billion, up 14% year-over-year. This performance continues to be broad-based with double-digit growth in almost every market we track. We also reached all-time revenue records for advertising, cloud services, music and payment services with December quarter records on the App Store and video. Company gross margin was at 48.2%, above the high end of our guidance range and up 100 basis points sequentially, driven by favorable mix and leverage. Products gross margin was 40.7%, up 450 basis points sequentially driven by favorable mix and leverage. Services gross margin was 76.5%, up 120 basis points sequentially, driven by mix.
Operating expenses landed at $18.4 billion, up 19% year-over-year. This was within the range we provided and driven by increased investment in R&D. Net income was $42.1 billion and diluted earnings per share was $2.84, up 19% year-over-year. Both net income and diluted EPS were all-time records and these incredibly strong business results drove an all-time record for operating cash flow coming in at $53.9 billion.
Now I'm going to provide some more details for each of our revenue categories. iPhone revenue was $85.3 billion, up 23% year-over-year, driven by the iPhone 17 family. iPhone saw strength around the world reaching all-time revenue records in many of the markets we track, including the U.S., Greater China, Latin America, Western Europe, the Middle East, Australia and South Asia as well as a December quarter record in India. The iPhone active installed base grew to an all-time high and set a new all-time record for upgraders in aggregate and across many countries, including the U.S., China Mainland, Japan and India.
According to a recent survey from World Panel, iPhone was a top-selling model in the U.S., Urban China, the U.K., Australia and Japan. Customers are loving the latest iPhone lineup. The latest customer satisfaction for the iPhone 17 family in the U.S. was measured at 99% by 451 Research. Mac revenue was $8.4 billion, down 7% year-over-year. As we described in the last call, we faced a very difficult compare against an M4 MacBook Pro, Mac mini and iMac launches in the year ago quarter. Despite this difficult compare, we continue to see growth in several emerging markets, including Brazil, India, Malaysia, Vietnam and more.
And as Tim mentioned earlier, the Mac installed base reached another all-time high with nearly half of the customers who purchased the Mac being new to the product. And in the U.S., customer satisfaction for Mac was measured at 97%. iPad revenue was $8.6 billion, up 6% year-over-year, driven by the M5 powered iPad Pro and the A16 powered iPad. We continue to add new users to the iPad. In fact, over half the customers who purchased the iPad during the quarter were new to the product. This helped the iPad installed base to reach an all-time high and we also reached an all-time high for upgraders. Based on the latest reports from 451 Research, customer satisfaction was 98% in the U.S.
Wearables, Home and Accessories revenue was $11.5 billion, down 2% year-over-year. During the quarter, we experienced constraints on the AirPods Pro 3, and we believe the overall category would have grown had it not been for these constraints. The Wearables installed base reached a new all-time high, with over half of the customers purchasing an Apple Watch during the quarter being new to the product. And in the U.S., customer satisfaction was recently reported at 96%.
Our services revenue reached an all-time high of $30 billion, up 14% year-over-year. As we said earlier, we had all-time revenue records on advertising, music, payment services and cloud services where we saw a double-digit growth on paid subscribers. We continue to be optimistic about the future of our services business. With our installed base of over 2.5 billion active devices, we have an incredibly strong foundation for new growth opportunities. We saw increased customer engagement across our service offerings with both transacting and paid accounts reaching all-time highs in the quarter.
And we continue to improve the quality and expand the breadth of our services offerings. From new wallet features like Digital ID, which provides a way for users to create an ID in wallet using information from their U.S. Passport to additional ads coming to search in the App Store, which provides advertisers more ways to drive downloads from search.
Turning now to enterprise. Organizations are continuing to expand their fleet of Apple devices to drive productivity while remaining secure. Snowflake has deployed over 9,000 Mac devices company-wide, establishing Mac as a primary laptop across all business units, resulting in increased performance and a reduction in support tickets. AstraZeneca is rolling out over 5,000 M5 powered iPad Pros to its pharmaceutical sales team to take advantage of AI capabilities, including Apple Intelligence, while meeting with clinicians daily. And in Mexico, Coppel, the country's largest domestic retailer recently added MacBook Air in addition to a growing fleet of over 10,000 iPad devices.
Let's turn to our cash position and capital return program. We ended the quarter with $145 billion in cash and marketable securities. We had $2.2 billion of debt maturities and decreased commercial paper by $6 billion, resulting in $91 billion in total debt. Therefore, at the end of the quarter, net cash was $54 billion. During the quarter, we returned nearly $32 billion to shareholders. This included $3.9 billion in dividends and equivalents and $25 billion through open market repurchases of 93 million Apple shares. As we move ahead into the March quarter, I'd like to review our outlook which includes the types of forward-looking information that Suhasini referred to.
Importantly, the color we're providing assumes that global tariff rates, policies and their application remain in effect as of this call, and the global macroeconomic outlook does not worsen from today. We expect our March quarter total company revenue to grow by 13% to 16% year-over-year, which comprehends our best estimates of constrained iPhone supply during the quarter.
We expect services revenue to grow at a year-over-year rate similar to what we reported in the December quarter. We expect gross margin to be between 48% and 49%. We expect operating expenses to be between $18.4 billion and $18.7 billion, which is at a similar level to what we reported in the December quarter and driven by higher R&D on a year-over-year basis. We expect OI&E to be around $100 million, excluding any potential impact from the mark-to-market of minority investments and our tax rate to be around 17.5%.
Finally, today, our Board of Directors has declared a cash dividend of $0.26 per share of common stock payable on February 12, 2026, to shareholders of record as of February 9, 2026.
With that, let's open the call to questions.
Question-and-Answer Session
Suhasini Chandramouli
Director of Investor Relations
[Operator Instructions] Operator, may we have the first question, please?
Operator
Certainly. We'll go ahead and take our first question from Amit Daryanani of Evercore.
Amit Daryanani
Evercore ISI Institutional Equities, Research Division
Yes, I have 2. Maybe to start with -- there's a lot of focus on the impact of memory to a host of companies, and I'd love to kind of get your perspective when you folks are guiding gross margins up into the March quarter. Just talk about, A, your comfort in securing the bits that you need for shipments? And B, how do we think about memory inflation flowing through Apple's model over time?
Timothy Cook
CEO & Director
Yes, Amit, it's Tim. Let me back up a bit and talk about the constraints that Kevan referred to in his remarks and memory and try to get both of these out at once. First of all, we were thrilled with the customer response on the latest iPhone lineup. It exceeded our expectations to say the least. And iPhone grew 23%. What the result of that was, was that we exited the December quarter with very lean channel inventory due to that staggering level of demand.
And based on that, we're in a supply chase mode to meet the very high levels of customer demand. We are currently constrained. And at this point, it's difficult to predict when supply and demand will balance. The constraints that we have are driven by the availability of the advanced nodes that our SoCs are produced on. And at this time, we're seeing less flexibility in supply chain than normal, partly because of our increased demand that I just spoke about.
From a memory point of view, to answer your question, memory had a minimal impact on the Q1, so the December quarter gross margin. We do expect it to be a bit more of an impact to the Q2 gross margin and that was comprehended in the outlook of 48% to 49% that Kevan gave earlier.
Beyond Q2, we don't obviously provide outlooks beyond the current quarter. But we do continue to see market pricing for memory increasing significantly. As always, we'll look at a range of options to deal with that. So hopefully, that gives you the full view.
Amit Daryanani
Evercore ISI Institutional Equities, Research Division
I appreciate all the clarity on that, Tim. Maybe the other -- second question I have for you is maybe just touch on the China strength you folks had. I think this is very close to all-time high revenues you've had in China. What's driving the strength over here? And just sort of the durability of the growth rate we saw in the December quarter would be helpful to understand.
Timothy Cook
CEO & Director
Sure. Greater China was up 38% year-on-year. It was driven by iPhone where we set an all-time revenue record. So it was the best iPhone quarter in history in Greater China. It's driven by the customer enthusiasm for the iPhone 17 lineup. And I would tell you that during the quarter, traffic in our stores in China grew by strong double digits year-over-year. It was a terrific quarter. Our installed base reached an all-time high in both Greater China and Mainland China. And we set an all-time record for the upgraders and we saw strong double-digit growth on switchers.
And according to a survey from Worldpanel, iPhones were the top 3 smartphones in urban China during the quarter. So it was -- and it's really driven primarily by the product strength and the customer response to the product strength. We do see on non-iPhone products that the majority of customers that are buying a Mac, an iPad, a watch are still new to that product. So that's a very good sign for us. And if you look at iPad on that same survey, iPad was the top tablet model in urban China.
And according to Counterpoint, the MacBook Air was the top-selling laptop model, and Mac mini was the top selling desktop model in the December quarter. So overall, a great quarter in China. We could not be more happy with it.
Suhasini Chandramouli
Director of Investor Relations
Awesome. Thank you, Amit. Operator, could we get the next question, please?
Operator
Our next question is from Erik Woodring of Morgan Stanley.
Erik Woodring
Morgan Stanley, Research Division
Tim, congrats on announcing the partnership with Google, and we're all excited to see what you bring to market later this year. When I think about your AI initiatives, it's clear there are added costs associated with that. We're obviously seeing that flow through in OpEx. Can you help us understand maybe what the revenue upside potential that exists with AI? Many of your competitors have already integrated AI into their devices. And it's just not clear yet what incremental monetization they're seeing because of AI, but you're always disciplined with investing. You obviously have a differentiated product. So how do you monetize AI? And what's the time line to realizing that ROI? Then a quick follow-up.
Timothy Cook
CEO & Director
Well, let me just say that we're bringing intelligence to more of what people love, and we're integrating it across the operating system in personal and private way. And I think that by doing so, it creates great value and that opens up a range of opportunities across our products and services. And we're very happy with the collaboration with Google as well, I should add.
Erik Woodring
Morgan Stanley, Research Division
Okay. And then maybe just a follow-up. Now that you have kind of more time and data to evaluate this cycle, can you maybe help us understand what the primary factors are driving strength in the iPhone? I'm sure there's a number of factors, but if you had to point to one or 2, just what would they be? And how sustainable do you think those are?
Timothy Cook
CEO & Director
I think it's different for different cohorts of where people are coming from in the device that they have. But it's a combination of things always that make the product sing. It's the display. It's the camera. It's the performance. It's the new selfie camera. It's the design. The design is beloved. And so it's all of these things that come together at once and have -- are producing a very strong product cycle as witnessed by our December quarter results.
Suhasini Chandramouli
Director of Investor Relations
Awesome. Thank you, Erik. Operator, could we get the next question, please?
Operator
We'll now go to Michael Ng of Goldman Sachs.
Michael Ng
Goldman Sachs Group, Inc., Research Division
Wonderful. I have 2 as well, if I could. First, it was encouraging to hear about the revenue growth outlook of 13% to 16% for the March quarter. I was just wondering if you could talk about any comps that we should be particularly aware of as we kind of think about each of the product categories? I know last year, you guys had MacBook Air with M4, the iPhone 16E, the iPad with A16 and the iPad Air with M3. So just wanted to ask if those things would create tough comps? Or is it just less of an issue just given the new product outlook?
Kevan Parekh
Senior VP & CFO
Yes, Mike, it's Kevan. Thanks for the question. Yes, I wouldn't say there's any particular comp issue that we'd note. As you recall, last quarter, we talked about the difficult comparison we had on Mac. But there's nothing that rises to that kind of color that we'd outlined in the outlook. And so I think it's just a continuation of the strong cycle. We're seeing subject to the constraints that I had mentioned in the prepared remarks, and that Tim alluded to a little earlier as well.
Michael Ng
Goldman Sachs Group, Inc., Research Division
Great. Wonderful. And then just on services, advertising, strong in the quarter. I wanted to ask about some of the new growth opportunities in advertising. I know you guys are doing the new ad slots in the App Store. Maybe you could just talk a little bit about that? And then any plans to do more in advertising across other products like Maps or TV?
Kevan Parekh
Senior VP & CFO
Yes, sure. Michael, what I would say, just if I step back in general, I think as we outlined, we saw really good broad-based performance in our across our services business, so ranging from all-time records in advertising, music, payment services and cloud services. So I think we see really good opportunities across a lot of our service categories. And we continue to add new service offerings. We talked about what we added to the wallet like Digital ID, and you referenced the additional kind of additional ads coming in the search in the App Store, which we are excited about. It provides advertisers more ways to be discovered.
And so I think we'll continue to look for ways to expand opportunities to add value to users and also create opportunities for Apple. I think as we talked about, we created -- crossed really significant milestone at 2.5 billion active devices. So we really feel excited about the opportunity that provides for our services business as well.
Suhasini Chandramouli
Director of Investor Relations
Thank you, Mike. Operator, could we get the next question, please?
Operator
The next question will be coming from Ben Reitzes of Melius.
Benjamin Reitzes
Melius Research LLC
First question is on Google partnership again. I wanted to understand how you came to that decision with regard to the AI and Siri in particular? And if there's an opportunity for you guys to share in revenue too with that partnership like you do in search?
Timothy Cook
CEO & Director
Yes. We basically determined that Google's AI technology would provide the most capable foundation for AFM -- I'm sorry, Apple Foundation Models. And we believe that we can unlock a lot of experiences and innovate in a key way due to the collaboration. We'll continue to run on the device and run in private cloud compute and maintain our industry-leading privacy standards in doing so. In terms of the arrangement with Google, we're not releasing the details of that.
Benjamin Reitzes
Melius Research LLC
Bummer. Okay, I tried. So yes, you knew it would be me. So the next question is on gross margin. I'm pretty shocked. I got to hand it to you, Tim, that you're able to do 48% to 49%. What's really going on there? How are you doing that with this memory, the NAND prices? Is it due to mix that there's a good -- less hardware and more services and services margins are going up? How are you doing it to go -- to keep it at 48% to 49%?
Kevan Parekh
Senior VP & CFO
Yes, Ben, this is Kevan. Let me start maybe by just reflecting on the Q1 gross margin. I think we talked about the fact that we landed at 48.2%. So just above the high end of the range that we provided on the last call. And I think if you look at that performance, we were up 100 basis points sequentially. We talked about the fact that we had favorable mix. I mean, as you know, when we have a good product cycle, a strong price cycle we're seeing for iPhone, that does lend itself to a bit more favorable opportunity on the mix and leverage side. So we're having a strong iPhone cycle, as Tim outlined. And so that also translated itself. We talked about products sequentially went up by 450 basis points.
So I think, in general, I think we're just seeing favorable mix dynamics as well. Services continues to contribute as well, that business is growing double digits. So that also is a contributor. And I think that as we looked at our guidance, we're providing a similar range to where we reported in December. And there's going to be a few puts and takes. We do expect to see favorable mix in the services. As you know, when we move from Q1 to Q2, that tends to be the case, and that's partly offset by a seasonal loss of leverage. So there will be puts and takes. But again, we feel pretty good about the guide of 48% to 49%, which is similar to the range we reported in December.
Suhasini Chandramouli
Director of Investor Relations
All right. Thanks, Ben. Operator, could we get the next question, please?
Operator
The next question will be coming from David Vogt of UBS.
David Vogt
UBS Investment Bank, Research Division
Maybe Tim or Kevan, if we could pull out a little bit. Can you help us understand how you're thinking about the overall kind of smartphone market demand, particularly given where memory prices are headed? And we've heard some conversations with some other OEMs as well as component providers that are worried about either the availability of components, potential market weakness in terms of demand destruction and some of the actions to offset our higher prices. I know you don't give outlooks for the full year, but how are you thinking about all of those different vectors and what that might mean for the overall smartphone market? And then ultimately, what that might mean for demand for iPhones as we move through the rest of this calendar year?
Timothy Cook
CEO & Director
Yes. On the supply side, I had made comments earlier about the constraint that we are seeing in Q2. We -- and that's reflected in the revenue guidance that Kevan gave earlier. The constraint, as I'd mentioned, is due to the advanced node capacity. And it's really a result of growing so well in Q1 with the 23% and having less flexibility partly due to that in the process to increase it as much as we would like to increase it. Beyond Q2, I don't really want to comment on supply. Supply is a function of a lot of things in the industry that move around a lot. So I wouldn't want to comment on that.
And I commented before on the -- on memory pricing. And so hopefully, that answers your question. In terms of smartphone demand, we believe that based on the information that we've got is we gained share in the December quarter. Obviously, the market wasn't growing at 23%. So we feel good about doing that. And -- but I don't -- I wouldn't want to predict how the market reacts in the future. It's very difficult to do that.
David Vogt
UBS Investment Bank, Research Division
Got it. At the risk of not getting this answered, I'm going to follow up with, can you maybe help us understand, you mentioned there's a range of options that you're looking at. How do -- how should we think about kind of like LTAs in the marketplace? I mean, is that an option as we move through the year? Or is it more spot-based from -- on a perspective, particularly around memory? Just want to get a better sense for how we should think about kind of the dynamics in the marketplace.
Timothy Cook
CEO & Director
It's a range. And so I don't want to get more specific than that. I mean there are different levers that we can push and who knows how successful they'll be, but there's just a range of options.
Suhasini Chandramouli
Director of Investor Relations
Great. Thank you, David. Operator, could we get the next question, please?
Operator
We'll now be taking questions from Wamsi Mohan from Bank of America.
Wamsi Mohan
BofA Securities, Research Division
Tim, on services, you grew a pretty impressive 14%. And I know you said that the App Store was a record for the December quarter. But third-party data is showing a notable deceleration in App Store growth, maybe 7% in the December quarter relative to your 14% growth. I was hoping if you could maybe confirm that. And secondarily, if it's correct, what might be some of the drivers of that? And what could be things that you could do to reverse that in future quarters? And I have a follow-up.
Kevan Parekh
Senior VP & CFO
Wamsi, it's Kevan here. Look, I think we want to reiterate the fact that during the December quarter, we had a quarterly record on the App Store. As you know, we don't provide specific color on how the individual services categories have done. But again, if we step back, I think we saw, again, broad-based growth across all the different categories, also across various geographies. We had all-time records in both developed and emerging markets as well. So -- and double-digit growth in both of those too. And so I think in general, we don't provide the color at the detailed services level.
Wamsi Mohan
BofA Securities, Research Division
Okay. I guess back to the memory price. I appreciate you have a range of options to address that. Historically, Apple has not used a pricing lever unless FX markets got maybe very dislocated to prevent arbitrage or issues like that. But given some of these unprecedented moves in memory, would pricing be a lever that you would be willing to pull or push outside of every other thing that -- outside of everything else that you can do?
Timothy Cook
CEO & Director
Yes, I wouldn't want to speculate on that one.
Suhasini Chandramouli
Director of Investor Relations
Thanks, Wamsi. Operator, could we have the next question, please?
Operator
We'll now go to Samik Chatterjee of JPMorgan.
Samik Chatterjee
JPMorgan Chase & Co, Research Division
Maybe for the first one, I'm just looking at your capital investment in the first quarter, which did moderate from the last one. And wondering if the partnership with Google on Gemini and sort of help collaboration to develop the next generation of Apple Foundation Models, does that have any near-term sort of impact on your intent to use Apple Private Cloud? I know you emphasized sort of the role Apple Private Cloud plays in the long term. But are there any changes on that front through this collaboration? Any thoughts around that? And I have a quick follow-up.
Kevan Parekh
Senior VP & CFO
Yes, sure. I think -- this is Kevan here. I think in general, as Tim outlined, we weren't going to provide any details on our arrangement and collaboration with Google. Just speaking of CapEx, in general, as you know, we have a hybrid model for CapEx. And so I think that what happens is our CapEx can be volatile, independent of kind of the volume and the performance of our business. And as you know, our CapEx is made of several different line items that include tooling, our facilities, retail investments -- investments in our retail store, data centers. And on tooling and data centers, we leverage this hybrid model that I mentioned before, which we leverage a combination of first and third-party capacity.
So in general, it's hard to read into the CapEx and draw any conclusions. And so I think I would just say there's going to be some ebbs and flows in CapEx. Last year, remember, we did build out our private cloud compute environment. And so we did have CapEx spending related to that in our results in December.
Samik Chatterjee
JPMorgan Chase & Co, Research Division
Got it. Got it. And my follow-up probably is for you again. You did mention product gross margin and the sort of drivers there for the product gross margin improvement. When you sort of highlighted mix as a driver, can you just sort of talk through what are the big differences in mix you're seeing for iPhone 17 versus 16? And did tariffs and tariffs coming in more favorable play a role at all and what you're expecting for tariffs for the next quarter?
Kevan Parekh
Senior VP & CFO
Yes. So there's a few things to unpack there. So on the overall margin on product side, I think I mentioned that we had favorable mix of products and leverage. I think given the strong iPhone cycle we're seeing, that was, I would say, probably a higher favorability than you might have seen in maybe other cycles. And as well, as you know, in Q1, typically, we do see the impact of the cost structure of our new products that we launch. And in this case, we are seeing a more favorable offset from mix of products and leverage versus historical -- sequential changes from Q4 to Q1. On the tariff piece, we had outlined an amount of $1.4 billion for the December quarter, and we landed roughly in that range at that level.
Suhasini Chandramouli
Director of Investor Relations
Awesome. Thank you, Samik. Operator, could we have the next question, please?
Operator
We'll now go to Krish Sankar of TD Cowen.
Sreekrishnan Sankarnarayanan
TD Cowen, Research Division
The first one I have was for Tim. I think you touched upon this earlier on the Gemini integration and Apple Foundation Models. How to think about kind of like the difference between Apple Foundation Models functionality and third-party models? Like does the Apple Foundation Models evolve to a different layer in the AI software stack? Or how to think about it as you partner with third-party frontier models? And I had a follow-up.
Timothy Cook
CEO & Director
Yes. Krish, you should think of it as a collaboration. And we'll obviously independently continue to do some of our own stuff. But you should think of what is going to power the personalized version of Siri is the collaboration with Google.
Sreekrishnan Sankarnarayanan
TD Cowen, Research Division
Got it. And then, a quick follow-up for maybe Kevan or Tim. Just a lot of discussion on memory pricing. Given that the memory constraint or commodity scarcity is impacting both the smartphone and the PC markets, and Apple, arguably having more purchasing power. Do you think this is a chance for you to increase your market share, both in iPhone and Mac at the expense of competition who might have more constraints in getting access to memory?
Timothy Cook
CEO & Director
Yes. I don't want to talk about kind of what has happened. And we do believe, as I had shared that iPhone gained share in the December quarter. And if you look at Mac for the full year of -- full calendar year '25, we also believe we gained share. And so we feel very good about our position.
Suhasini Chandramouli
Director of Investor Relations
Thank you, Krish. Operator, could we have the next question, please?
Operator
We'll now go to Atif Malik calling from Citi.
Atif Malik
Citigroup Inc., Research Division
The first one for Tim. Tim, some of the industry pundits are comparing the iPhone 17 upgrade cycle to the 2020, 2021 years as some of the iPhone 12, 13 users upgrade. Curious if you agree with that view. And also, if you can layer on the impact from Apple Intelligence to the refresh rate.
Timothy Cook
CEO & Director
I think each iPhone cycle has its own unique characteristics. And so I wouldn't compare it to a specific one. I think iPhone 17, the family of 17 is a unique product that brings several very compelling features in one product, and it has done extremely well. And so we feel quite good about it.
Kevan Parekh
Senior VP & CFO
Atif, I'll just add to Tim's comment that we talked about the fact we have a large and diverse installed base of customers. And so this product has really resonated with multiple cohorts, whether you're on older devices or newer iPhones as well. So we've seen really strong reaction to the product lineup.
Atif Malik
Citigroup Inc., Research Division
Great. As my follow-up, there was a lot of discussion on supply constraints. And I'm surprised that you guys are constrained on advanced packaging as you generally get your share at the big foundry. How long will these supply impact your ability to ship through demand?
Timothy Cook
CEO & Director
It's difficult to estimate demand when you haven't met the demand. And so we've -- obviously, we have internal estimates on that, but I don't want to share those. But it's very difficult. And just to be clear, it's the advanced nodes that we -- like 3-nanometer to be specific, where our SoCs or the latest SoCs are produced on as to what is gating the Q2 supply. And it's a direct result of the 23% growth, and that far outstripping what we had internally estimated and having more limited flexibility in the supply chain for some period of time. But I don't want to estimate when supply and demand will balance at this point.
Suhasini Chandramouli
Director of Investor Relations
Thank you, Atif. Operator, could we have the next question, please?
Operator
The next question will be coming from Aaron Rakers coming from Wells Fargo.
Aaron Rakers
Wells Fargo Securities, LLC, Research Division
I have 2 as well, and I'll try and stay away from the memory question. I'm curious -- and obviously, a lot of focus on the China demand, but I'm curious, you also called out India. And so can you maybe unpack some of the things that you're seeing in the Indian market as far as iPhone traction? Any kind of color on what is a very large installed base in India that seems to be a good growth opportunity for Apple still?
Timothy Cook
CEO & Director
Yes. Thanks for the question. We did set a quarterly revenue record during the December quarter. And to go a little further down, we set quarterly revenue records on iPhone and Mac and iPad and an all-time revenue record on services. So it was a terrific quarter in India. We really like what we see there. It's the second largest smartphone market in the world and the fourth largest PC market. So -- and we still have, despite a very nice growth history, we have modest share there. And so we think there's a huge opportunity for us there. And we could not be more excited about it.
If you look at the other thing that I would point out is that the majority of customers that are buying iPhone and Mac and iPad and watch are all new to that product. And so it speaks very well to opportunity there.
Kevan Parekh
Senior VP & CFO
Yes. And Aaron, I'd add, you mentioned the installed base. We're seeing strong double-digit growth in the installed base in India as well, which is really encouraging.
Aaron Rakers
Wells Fargo Securities, LLC, Research Division
Yes. And then as a quick follow-up, kind of tied to memory, maybe not so much. But part of this current generation iPhone cycle is you clearly deepened some of your own internal silicon capabilities on the device. I'm curious if that -- if we should think about that as a lever and maybe a supportive factor to gross margin that might be underappreciated? Any thoughts on where we go from here as far as continued opportunities of internalizing your own silicon?
Timothy Cook
CEO & Director
Yes. I'll let Kevan talk about the gross margin. But in terms of the product, which is at the heart of what we think about in the user. Apple silicon has just been an incredible game changer for us, starting with iPhone and then on iPad and of course, the Mac as of a few years ago. And so we believe it's a game changer and a major competitive advantage.
Kevan Parekh
Senior VP & CFO
Yes. And as far as impact on gross margin, we have been, as you know, investing in core technologies like our own silicon, our own modem. And certainly, while those do provide opportunities for cost savings and can be reflected in margins, they also importantly provide the differentiation that's really important for our products as well and give us more control of our road map. So I think there's a lot of strategic value to it, but also we are seeing investments in our core technologies impacting gross margin in a positive way.
Suhasini Chandramouli
Director of Investor Relations
Awesome. Thank you, Aaron. Operator, could we have our last question, please?
Operator
Most certainly. Our last question will be coming from Richard Kramer calling from Arete Research.
Richard Kramer
Arete Research Services LLP
I have 2 questions. Tim, when you think about how Apple might manage AI, do you see that evolving towards more edge AI or on device services versus cloud-based AI? And are you confident you've reserved sufficient data center capacity to support the widespread Siri adoption, especially given that you're not following the other hyperscalers and sharply increasing CapEx?
Timothy Cook
CEO & Director
The answer is that we see both being important, the on-device and the private cloud compute. And so we don't see it as an either/or we see it as both. And we believe it's a differentiator because of our privacy approach. In terms of do we have enough capacity, it's hard to estimate with precision what the demand will be. But we've done sort of the best job that we can do and either have or are putting capacity in for it.
Richard Kramer
Arete Research Services LLP
Okay. And you also -- you mentioned the 2.5 billion active device number but Apple Intelligence features have only been available since the 15 Pro. So can you speak at all to roughly what portion of your iPhone or overall active device installed base is now AI capable? And has this been a factor in maybe a more gradual pace of launching wider AI services?
Kevan Parekh
Senior VP & CFO
Yes, Richard, this is Kevan. We don't provide that specific number, but it is a growing number, as you can imagine in our installed base. And so we're encouraged by the amount of devices now that are capable, but we're not going to provide a specific figure on that today.
Suhasini Chandramouli
Director of Investor Relations
All right. Thank you, Richard. A replay of today's call will be available for 2 weeks on Apple Podcasts, as a webcast on apple.com/investor and via telephone. The number for the telephone replay is (866) 583-1035. Please enter confirmation code 890-2968 followed by the pound sign. These replays will be available by approximately 5:00 p.m. Pacific Time tonight. Members of the press with additional questions can contact Josh Rosenstock at (408) 862-1142. And financial analysts can contact me, Suhasini Chandramouli, with additional questions at (408) 974-3123. Thanks again for joining us today.
Operator
Once again, this does conclude today's conference. We do appreciate your participation.

"""

In [39]:
text2 = """Apple Inc. (AAPL) Q2 2026 Earnings Call April 30, 2026 5:00 PM EDT
Company Participants
Suhasini Chandramouli - Director of Investor Relations
Timothy Cook - CEO & Director
John Ternus - Senior Vice President of Hardware Engineering
Kevan Parekh - Senior VP & CFO
Conference Call Participants
Erik Woodring - Morgan Stanley, Research Division
Benjamin Reitzes - Melius Research LLC
Michael Ng - Goldman Sachs Group, Inc., Research Division
Wamsi Mohan - BofA Securities, Research Division
Amit Daryanani - Evercore ISI Institutional Equities, Research Division
David Vogt - UBS Investment Bank, Research Division
Samik Chatterjee - JPMorgan Chase & Co, Research Division
Aaron Rakers - Wells Fargo Securities, LLC, Research Division
Presentation
Suhasini Chandramouli
Director of Investor Relations
Good afternoon, and welcome to the Apple Q2 Fiscal Year 2026 Earnings Conference Call. My name is Suhasini Chandramouli, Director of Investor Relations. Today's call is being recorded. Speaking first today is Apple's CEO, Tim Cook. John Ternus will be joining after that for a brief set of remarks, and he'll be followed by CFO, Kevan Parekh. After that, we'll open the call to questions from analysts.
Please note that some of the information you'll hear during our discussion today will consist of forward-looking statements, including, without limitation, those regarding revenue, gross margin, operating expenses, other income and expense, taxes, capital allocation and future business outlook. These statements involve risks and uncertainties that may cause actual results or trends to differ materially from our forecast, including risks related to the potential impact to the company's business and results of operations from macroeconomic conditions, tariffs and other measures and legal and regulatory proceedings.
For more information, please refer to the risk factors discussed in Apple's most recently filed reports on Form 10-Q and Form 10-K and the Form 8-K filed with the SEC today, along with the associated press release. Additional information will also be in our report on Form 10-Q for the quarter ended March 28, 2026, to be filed tomorrow and in other reports and filings we make with the SEC. Apple assumes no obligation to update any forward-looking statements, which speak only as of the date they are made.
I'd now like to turn the call over to Tim for introductory remarks.
Timothy Cook
CEO & Director
Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. Before we get into the quarter, I wanted to take a moment to talk about the transition we recently announced. I just celebrated my 28th anniversary of being here at Apple, 15 years as CEO. In fact, this will be my 89th earnings call. I'll always be proud of the impact Apple has had on our users' lives, and I can't begin to express how grateful I am for our amazing teams. It's because of them that there is no company like Apple, and I truly believe there never will be. This moment for the transition is the right one for a number of reasons. First, our business has been performing extremely well. The first half of this year was very strong, growing double digits year-over-year. Second, our road map is incredible. And most importantly, we have the right leader ready to step into the role.
As I have said, there is no one on this planet I trust more to lead Apple into the future than John Ternus. John is a brilliant engineer, a deep thinker, a person of remarkable character and a born leader. I know he will push us to go further than we think is possible in order to deliver the greatest products and services for our users. I have been so proud to call him a colleague and a friend, and I will be even more proud to call him Apple's CEO.
Over the coming months, John and I will be working closely together to make sure this transition is perfectly smooth. I very much look forward to stepping into the role of Executive Chairman on September 1. As I've told John, I will be here to support him in any way he needs and in any way I can. I'm incredibly optimistic about Apple's future, and I know we have the right team in place to deliver on the promise of this company. I also want to take just a moment to share my profound gratitude for our shareholders, especially our long-term shareholders, for believing in Apple and for your support over the years. It means a great deal to all of us.
With that, I'd like to bring John on the call for a moment to say a few words. John?
John Ternus
Senior Vice President of Hardware Engineering
Thanks, Tim, and thanks to everyone on the call. In my view, Tim is one of the greatest business leaders of all time. Stepping into the role of CEO is an incredible honor, and it means a great deal to me to have Tim's trust and confidence. I want to echo Tim's sentiment about our shareholders, especially those who have been with us for many years. Thank you so much for your confidence in our company. As you know, one of the hallmarks of Tim's tenure has been a deep thoughtfulness, deliberateness and discipline when it comes to the financial decision-making of the company. And I want you to know that it's something Kevan and I intend to continue when I transition into the role in September. This is an especially exciting moment for Apple.
As Tim mentioned, we have an incredible road map ahead. And while you're not going to get me to talk about the details of that road map, suffice it to say, this is the most exciting time in my 25-year career at Apple to be building products and services. There are so many opportunities before us, and I couldn't be more optimistic about what's to come. For now, let me simply say I am deeply grateful to Tim, to the executive team and to everyone at Apple, and I look forward to all of the important work ahead.
And with that, let me turn it back over to Tim.
Timothy Cook
CEO & Director
Thanks, John. Now let me turn to the quarter. Today, Apple is proud to report $111.2 billion in revenue, up 17% from a year ago and a March quarter record, which was above the high end of our guidance range despite supply constraints. Customer enthusiasm for iPhone has been extraordinary with revenue growing 22% year-over-year to achieve a March quarter record. Services reached an all-time revenue record, growing 16% from a year ago, while EPS set a March quarter record of $2.01, up 22% year-over-year. We set March quarter revenue records and grew double digits in every geographic segment, including strong double-digit growth in Greater China and the rest of Asia Pacific. We also achieved March quarter revenue records in both developed and emerging markets and saw double-digit growth in nearly every emerging market we track, including India.
We recently marked Apple's 50th anniversary with celebrations in our retail stores and with users around the world. It was a special moment for us to reflect on the incredible journey we've shared with our users to thank everyone who's been a part of it and to look forward to writing the next chapter in our story of innovation. We have always believed that people who think different can change the world, and we have been proud to build tools and technologies that allow them to do just that.
In March, we put an amazing showcase of human creativity and ingenuity in action with updates across iPhone, iPad and Mac. Through an unforgettable week of innovation, we also unveiled MacBook Neo, giving us an opportunity to bring the power of Mac to more people than ever before. I'll have more to say on that and all the incredible things we delivered for our customers over the last few months.
Now let's take a closer look at results from across our product line, beginning with iPhone. As I mentioned earlier, iPhone had an excellent quarter with $57 billion in revenue, a March quarter record despite supply constraints. During the quarter, we welcomed iPhone 17E, the newest addition to what is already the strongest iPhone lineup we've ever had. It brings outstanding performance and core iPhone experiences at a remarkable value for everyone from enterprise teams to consumers. Across the lineup, this is the most powerful, capable and versatile iPhone family we've ever created. That starts with the latest in Apple silicon for iPhone, A19 and A19 Pro, which include neural accelerators in the GPU to deliver a huge boost to AI performance.
With incredible performance and battery life and deep integration of Apple Intelligence, iPhone continues to set the standard for what a smartphone can be. Customers are capturing stunning photos and videos with our most advanced camera system ever on iPhone 17 Pro and Pro Max, including an 8x optical quality zoom and the all-new Center Stage front camera, unlocking entirely new ways to frame, create and share their moments. In fact, during the recent mission, Artemis II astronauts captured some truly otherworldly images of earth and space using iPhone 17 Pro Max. Meanwhile, iPhone Air users are tapping into the Pro level performance in our slimmest iPhone ever. And with iPhone 17, we're seeing a strong response not only from customers upgrading from previous generations, but also from people choosing iPhone for the very first time.
We've been enormously pleased with how the entire lineup has been received. In fact, the iPhone 17 family is now the most popular lineup in our history when looking at the launch through the March quarter. And according to IDC, we gained market share during the quarter. Mac revenue was $8.4 billion for the March quarter, up 6% from a year ago despite supply constraints driven by higher-than-expected levels of demand. We're delighted with the reception of what is the most advanced Mac lineup in our history. We set March quarter records for upgraders and customers new to Mac. And according to IDC, we gained market share in the quarter.
From Mac Mini to MacBook Pro and everything in between, Mac is the best platform for AI with Apple Silicon delivering exceptional performance, industry-leading efficiency and the ability to run advanced models locally in ways that simply weren't possible before. It's so exciting to see how strongly users are embracing Mac for these capabilities. There's tremendous enthusiasm for MacBook Neo, which made its debut during the March quarter, opening up an entirely new way to experience Mac at a breakthrough price.
We've also further improved MacBook Air, already the world's most popular laptop with M5, making everyday tasks faster and more responsive than ever. MacBook Pro reaches new heights with M5 Pro and M5 Max, delivering extraordinary performance and dramatically advancing what users can do with AI on a portable system. And for desktop users, Studio Display pairs beautifully with Mac, while the all-new Studio Display XDR takes things even further, bringing unmatched image quality and an extraordinarily immersive experience to Pro workflows.
Turning to iPad. Revenue was $6.9 billion, up 8% from a year ago. iPad continues to be a great choice for students, small business owners, artists and so many others because it empowers entirely new ways to work, learn, create and connect. It's not just about mobility. It's about versatility, delivering a uniquely flexible experience that adapts to whatever users want to accomplish. Today, our iPad lineup is stronger than ever, led by the arrival of the M4-powered iPad Air. With a remarkable leap in performance, it raises the bar for what users can do on iPad from advanced creative workflows to powerful productivity and immersive learning. And with the addition of our latest Apple silicon, along with the N1 wireless networking chip and C1X modem, users can stay seamlessly connected wherever they are.
Across Wearables, Home and Accessories, revenue for the March quarter came in at $7.9 billion, up 5% from a year ago. Apple Watch Ultra 3, Apple Watch Series 11 and Apple Watch SE continue to play an essential role in users' lives going far beyond fitness tracking to deliver meaningful insights and support for their health and well-being. From helping users stay active and reach their fitness goals to delivering powerful science-backed health insights that can prompt meaningful conversations with care providers, Apple Watch is with them every step of the way. It's tremendously meaningful to see how Apple Watch continues to empower users to better understand their health, make more informed decisions and in many cases, change and even save lives.
During the quarter, we introduced customers to a new level of audio experience with AirPods Max 2, delivering stunning sound quality and our most advanced active noise cancellation yet. At the same time, AirPods Pro 3 combined an incredibly immersive listening experience with intelligent features that adapt to how users move, train and live. And whether it's a call across town or a conversation across continents, AirPods make it effortless to stay connected. AirPods can bridge languages too, thanks to Live Translation powered by Apple Intelligence. In addition to live translation, Apple Intelligence brings together dozens of powerful capabilities from visual intelligence to cleanup and photos that are seamlessly integrated into the moments that matter most to our users every day. And we look forward to bringing a more personalized Siri to users coming this year.
What truly sets Apple apart is how Apple Intelligence is woven into the core of our platforms, powered by Apple Silicon and designed from the ground up to deliver intelligence that is fast, personal, and private. This is not AI as a stand-alone feature, but AI as an essential intuitive part of the experience across our devices. It builds on years of innovation from the neural engine to advanced on-device processing, enabling capabilities that are not only incredibly powerful, but also respectful of user privacy.
Increasingly, that same foundation is drawing developers and researchers to our products as powerful platforms for building and running Agentic AI, thanks to the unique combination of performance, efficiency and on-device capabilities. When you combine this level of integration with our relentless focus on the customer experience, it becomes clear why Apple platforms are the best place to experience AI.
Now let's turn to services, which set an all-time revenue record with $31 billion. We saw double-digit growth in both developed and emerging markets and set new all-time revenue records across most of the services categories. There's no better place to find celebrated storytellers than Apple TV. Audiences are applauding the return of shows like your friends and neighbors, shrinking and for all mankind while discovering new favorites like Widow's Bay. Apple TV has also earned its place among the most decorated names in entertainment with more than 800 wins and more than 3,400 nominations in the 6 years since launch.
This is a great time for sports fans on Apple TV too. Formula 1 season kicked off in March and Apple TV subscribers in the U.S. have one of the best views of the track. The new MLS season is also well underway and subscribers in more than 100 countries and regions can watch every match with no blackouts. And Friday Night Baseball returned for its fifth year on Apple TV with a full season of marquee matchups.
In retail, we had a March quarter revenue record and saw very high levels of store traffic throughout the quarter. From New York to Chengdu to Paris, it was wonderful to see stores around the world at the center of Apple's 50th anniversary celebrations. We were also thrilled to open the doors to our sixth store in India. It has been wonderful to see how we've continued to grow in India in recent years, part of our larger efforts to connect with even more customers in emerging markets all over the world.
At Apple, we believe powerful innovation and uncompromising quality can go hand-in-hand with sustainability. Over the last year, we've reached new milestones in the environment, including the use of recycled content in 30% of the materials in all of our products shipped in 2025, the most we've ever had. That includes the use of 100% recycled cobalt in all Apple design batteries and 100% recycled rare earth elements in all magnets. We've also achieved our goal of removing plastic from packaging with every Apple product now shipping in fiber-based packaging. All of this is a testament to the outstanding forward-thinking and innovative work of our teams.
We're also making great progress in advancing American supply chain innovation. As part of our $600 billion commitment to the U.S., we were pleased to share recently that Mac mini production is coming to America later this year, expanding our factory operations in Houston with a brand-new facility. In March, we were thrilled to welcome 4 new companies to our American manufacturing program to help manufacture essential materials and components for Apple products sold worldwide. These include sensors that support key iPhone features like camera stabilization and integrated circuits essential for features like crash detection and activity tracking. These efforts build on the progress we've made in the American manufacturing program, including the work we're doing to advance an end-to-end silicon supply chain across the U.S.
At TSMC's Arizona facility, for example, Apple is on track to purchase well over 100 million advanced chips. As we're accelerating our long-standing support for U.S. innovation, we're also investing in America's workforce. We're looking forward to opening the doors to an all-new advanced manufacturing center in Houston later this year, which will provide hands-on training led by Apple experts and tailor-made for students, supplier employees and American businesses. Whether around the world or in our own backyard, we're proud of the difference Apple has made to enrich lives and support the communities we serve.
Looking ahead, we're delighted to welcome developers back to Apple Park for WWDC26. We can't wait to share what we've been working on from AI advancements to exciting new software and developer tools. It's going to be an incredible week. As always, we remain in relentless pursuit of even more powerful innovations guided by our North Star, our users. As we celebrated 50 years of Apple, we are even more excited and more optimistic about the next 50 years and beyond.
With that, I'll turn it over to Kevan.
Kevan Parekh
Senior VP & CFO
Thanks, Tim, and good afternoon, everyone. Our revenue of $111.2 billion was up 17% year-over-year, a March quarter revenue record. We saw strong performance around the world with March quarter revenue records in every geographic segment. Foreign exchange was about a 2.5 percentage point tailwind to the March quarter growth rate. We also faced supply constraints on iPhone and to a lesser extent on Mac. We believe if we remove the favorable benefit from foreign exchange and add back the unfavorable impact from supply constraints, we would have had a higher growth rate for total company revenue for the quarter.
Products revenue was $80.2 billion, up 17% year-over-year, driven by double-digit growth in iPhone, setting a new March quarter record. Our installed base of over 2.5 billion active devices has reached another all-time high across all major product categories and geographic segments. Services revenue was $31 billion, up 16% year-over-year. We saw strong performance across the board with double-digit growth in the vast majority of the markets we track. Company gross margin was 49.3%, above the high end of our guidance range and up 110 basis points sequentially.
Products gross margin was 38.7%, down 200 basis points sequentially. Services gross margin was 76.7%, up 20 basis points sequentially. Operating expenses landed at $18.9 billion, up 24% year-over-year. This was slightly above the high end of our guidance range due to a one-time expense in SG&A. Net income was $29.6 billion and diluted earnings per share was $2.01, up 22% year-over-year. Both net income and diluted EPS achieved March quarter records and drove a very strong level of operating cash flow at $28.7 billion.
Now I'm going to provide some more details for each of our revenue categories. iPhone revenue was $57 billion, up 22% year-over-year, driven by the iPhone 17 family. iPhone grew double digits in the majority of markets we track, including the U.S., Latin America, Greater China, Western Europe, India, Japan, and Southeast Asia. The iPhone active installed base grew to an all-time high, and we set March quarter record for iPhone upgraders. According to a recent survey from World Panel, iPhone was a top-selling model in the U.S., Urban China, the U.K., Australia and Japan. We have been extremely pleased with the positive reception of the iPhone 17 family.
In fact, customer satisfaction for the iPhone 17 family in the U.S. was recently measured at 99% by 451 Research. Mac revenue was $8.4 billion, up 6% year-over-year, driven by the strength of the recent product launches, including MacBook Neo. We grew in both developed and emerging markets with double-digit growth in many emerging markets, including India and Indonesia. As Tim mentioned earlier, we had a March quarter record for customers new to the Mac, and this helped drive a new all-time record for the overall Mac installed base. And in the U.S., customer satisfaction for Mac was recently reported at 97%. iPad revenue was $6.9 billion, up 8% year-over-year, driven by the continued strength of the A16-powered iPad and the M5-powered iPad Pro.
The iPad installed base reached a new all-time high as iPad continued to reach new customers around the world. During the quarter, over half of the customers who purchased an iPad were new to the product. Many of these customers are in our emerging markets, where we grew iPad revenue by double digits, including in India, Mexico, and Thailand. And based on the latest reports from 451 Research, customer satisfaction was 98% in the U.S. Wearables, Home and Accessories revenue was $7.9 billion, up 5% year-over-year, driven by strength in wearables and accessories. We were pleased to see strength in our emerging markets where we set a new March quarter revenue record.
The wearables installed base reached a new all-time high with over half of the customers purchasing an Apple Watch during the quarter being new to the product. And in the U.S., customer satisfaction on Apple Watch was measured at 96%. Our services revenue reached an all-time high of $31 billion, up 16% year-over-year. The strong performance was broad-based with all-time records in both developed and emerging markets. And as Tim mentioned, we also set all-time revenue records in most of the services categories. We are optimistic about the future of our services business with our large installed base of over 2.5 billion active devices, we have an incredibly strong foundation for growth opportunities.
Both transacting and paid accounts reached new all-time highs in the quarter as we continue to see more customers leveraging our services offerings. And we continue to improve the quality and expand the breadth of our services from the expansion of features like Tap to Pay, now available in over 50 markets to deeper support for enterprise customers. Building on this, we launched Apple Business, a new all-in-one platform that combines our hardware, software and enterprise services, enabling companies to efficiently manage their deployments and scale their business. We continue to see more organizations in enterprise choosing Apple's devices for performance and productivity. Marsh, a leading professional services firm, deployed a large-scale refresh of corporate devices to iPhone 17 as part of a commitment to security alongside adopting Mac for internal AI development.
With Apple Silicon and its powerful unified memory architecture, leading AI developers like Perplexity are choosing Mac as their preferred platform to build enterprise-grade AI assistants that power autonomous agents and boost workplace productivity. Across the Mac lineup, customers are finding the right device for their needs. From MacBook Pro and MacBook Air to our newest addition, MacBook Neo, which delivers an unprecedented combination of quality, value and industry-leading security that is resonating strongly in enterprise and education. Kansas City Public Schools, for example, is switching their high school students from Windows laptops and Chromebooks to MacBook Neo, completing their transition to an all- Apple district. And in India, leading enterprise software provider, Freshworks, deployed over 5,000 MacBook Pro and MacBook Air to accelerate their AI development.
Let's turn to our cash position and capital return program. We ended the quarter with $147 billion in cash and marketable securities. We had $5.8 billion of debt maturities and commercial paper remained unchanged at $2 billion, resulting in $85 billion in total debt. Therefore, at the end of the quarter, net cash was $62 billion. During the quarter, we returned $15 billion to shareholders. This included $3.8 billion in dividends and equivalents and $11 billion through open market repurchases of 42 million Apple shares. Our repurchase activity at any time can be affected by a number of factors that we take into account. And as you're aware, we recently announced a CEO transition.
Taking a step back, we plan to continue our capital allocation philosophy of, first, making all the necessary investments needed to support the business and then returning excess cash to shareholders over time. Net cash neutral has been a valuable framework for our capital structure. And since 2018, we have significantly rightsized our balance sheet and reduced net cash by over $100 billion. As we move ahead, we are no longer providing net cash neutral as a formal target, and we will independently evaluate cash and debt. Capital returns will continue to be important to our overall approach of delivering long-term shareholder value.
Accordingly, our Board has authorized an additional $100 billion for share repurchases, and we're also raising our dividend by 4% to $0.27 per share of common stock. This cash dividend will be payable on May 14, 2026, to shareholders of record as of May 11, 2026. As we move ahead into the June quarter, I'd like to review our outlook, which includes the types of forward-looking information that Suhasini referred to. Importantly, the color we're providing assumes that global tariff rates, policies and their application remain in effect as of this call and the global macroeconomic outlook does not worsen from today. We expect our June quarter total company revenue to grow by 14% to 17% year-over-year, which comprehends our best view of constraint supply.
On iPad, keep in mind, we face a difficult compare driven by the launch of A16-powered iPad in the prior year. We expect services revenue to grow at year-over-year rate similar to what we reported in the March quarter after removing the favorable year-over-year impact from foreign exchange tailwinds. Keep in mind, during the March quarter, FX was a 2.5 percentage point tailwind to the total company growth rate. And for services, that impact was slightly more favorable. We expect gross margin to be between 47.5% and 48.5%. We expect operating expenses to be between $18.8 billion and $19.1 billion. We expect OI&E to be around $250 million, excluding any potential impact from the mark-to-market of minority investments and our tax rate to be around 17%.
With that, Tim and I will take questions.
Question-and-Answer Session
Suhasini Chandramouli
Director of Investor Relations
Thank you, Kevan. [Operator Instructions] Operator, may we have the first question, please?
Operator
Certainly. We'll go ahead and take our first question from Erik Woodring with Morgan Stanley.
Erik Woodring
Morgan Stanley, Research Division
And Tim, I'll save the congrats on the Art of war for next quarter, but it's been a pleasure working together. I would love, maybe, Tim, if I could ask you just to maybe contextualize the supply constraints you alluded to in your prepared remarks, meaning how much did demand outpace supply for iPhone and Mac in the March quarter? And does your June quarter guidance also reflect supply constraints for those segments? Or is that kind of an unconstrained guide as you see it today? And then a quick follow-up, please.
Timothy Cook
CEO & Director
Yes. Erik, thanks for your comments. We were constrained during the March quarter. This was primarily on iPhone and to a lesser extent on the Mac. And as we talked about in the last call, the constraints were primarily driven by the availability of the advanced nodes our SoCs are produced on. you look forward to the June quarter, the majority of our supply constraints will be on several Mac models given the continued high levels of demand that we're seeing. And we have less flexibility in the supply chain than we normally would.
For Mac, in the June quarter, there's 2 factors that are driving the constraints. One is that on the Mac Mini and the Mac Studio, both of these are amazing platforms for AI and Agentic tools. And the customer recognition of that is happening faster than what we had predicted. And so we saw higher-than-expected demand. The second reason is that the customer response to Mac Neo has just been off the charts, with higher-than-expected demand. And the March quarter record for customers -- we set a March quarter record for customers new to the Mac, partly due to the Neo. We think looking forward that the Mini and the Mac Studio may take several months to reach supply-demand balance. And so hopefully, that gives you a view of both Q2 and Q3 on the supply side.
Erik Woodring
Morgan Stanley, Research Division
All right. Awesome. And then Kevan, I'd love to maybe turn to you and kind of a surprise little announcement there talking about net cash neutral still a great path, but we're no longer providing this as a formal target. Could you maybe expand on that a bit? Are we thinking about any different type of capital return policy? It doesn't seem so, but maybe give a little bit more detail when you talk about making investments. Is that organic versus inorganic? Just maybe tease that comment out a little bit more for us would be super helpful.
Kevan Parekh
Senior VP & CFO
Yes, sure, Erik. Thanks for the question. Yes, let me just kind of reiterate what we said, which is really kind of more of a comment on the capital structure. But our goal of net cash neutral has really served us well. It's been a valuable framework for us and for our capital structure since 2018. We believe we're at a stage where we're evaluating cash and debt independently is really the right approach for us and allows us to make more optimal economic decisions around how we best utilize our debt and cash portfolios to support the business based on business factors and market conditions. We also believe we can manage this flexibility while also being very efficient and remaining disciplined. So with all that being said, we remain very committed to returning excess cash to shareholders.
And as we talked about our investment in the business, I think, as you know, we invest in the business, first and foremost, and then look to kind of return excess cash to shareholders. I think we have a very good track record of being disciplined. We've returned over $1 trillion to shareholders from the start of the program, over $850 billion of which has been through share repurchases. And so -- and the other piece as well that's really important is as part of that, we also have increased our buyback authorization by another $100 billion, and that's on top of the leftover capacity from the prior authorization. So you can see the capital return piece is something very important to us. And as we talked about in the prepared remarks, important to the overall approach to delivering long-term shareholder value.
Operator
Our next question is from Ben Reitzes with Melius Research.
Benjamin Reitzes
Melius Research LLC
I'll ask myself. The first one is there's just been a lot of talk, and it's great to, by the way, speak with you, Tim and John and Kevan. The first question is around -- there's been some commentary around an Agentic smartphone. By the way, I don't even know what that means, but there's comments that about AI on the edge and that agents could catalyze smartphones, but also shift the smartphone kind of form factor or maybe not. I was just wondering with the rise of agents, how you would like us to think about that? Is -- does this mean there's new products coming of a totally new form factor? Or does it change the game? Or anything high level you might want to say about that and that trend or potential nontrend?
Timothy Cook
CEO & Director
Ben, it's Tim. We don't get into our future road map. And so I don't want to give too much info there. But I would just say that we're thrilled with how the iPhone is doing, growing 22% in the quarter and followed up from an incredible Q1 and having the strongest cycle that we've ever had in our history from the launch through March quarter. We could not be happier with it.
Benjamin Reitzes
Melius Research LLC
Okay. Well, I appreciate that. I'm sure we'll hear a lot more. Then with regard to, I guess, the question around constraints and whatnot. And Tim, I may push you one more time, try to do it nicely, though, just given my age. The big concern out there is maybe how margins go after the June quarter given the components and trends and whatnot and all these constraints. I mean, is there some kind of overarching philosophy that you want us to think about? Do you feel -- and maybe Kevan wants to weigh in on this, do you see a lot of variability in the model? Or is 47%, 48% kind of a range you think you might be able to stay in? Or is there just no visibility beyond June to answer this question? I think any comfort level there as we go throughout the calendar year would be so helpful.
Timothy Cook
CEO & Director
Yes. Ben, let me talk about memory specifically, which I think is the root of the question. So -- and I'll go back to December for a moment and just walk you through the chronology. In the December quarter, we really had a minimal impact due to memory, and you can kind of see that in the gross margin results. We said it would be a bit more in the March quarter, and we did see higher memory costs in the March quarter, and they were partially offset by benefits from carry-in inventory that we had.
For the June quarter and what's embedded in the guidance that Kevan went through earlier, we expect significantly higher memory costs. They are also partly offset by the benefit of carry-in inventory. And then where we don't give color beyond June, I can tell you that beyond the June quarter, we believe memory costs will drive an increasing impact on our business. And we'll continue to evaluate this. And as we've said before, we'll look at a range of options.
Operator
Our next question comes from Michael Ng with Goldman Sachs.
Michael Ng
Goldman Sachs Group, Inc., Research Division
I have 2 as well. First, given the success of the MacBook Neo, I was wondering if you could talk a little bit about how it's helped drive penetration with new customer segments, whether that be education or value or emerging markets? And then how do you think about opportunities in underpenetrated markets more broadly? And how will your future product road map inform that strategy?
Timothy Cook
CEO & Director
Yes. Right now, we're supply constrained on the MacBook Neo. The response has been -- we were very bullish on the product before announcing it and -- but we undercalled the level of enthusiasm that we'd be with it. And it's very much focused on getting the Mac to even more people than we were reaching before. We're very focused on customers new to the Mac and customers that have been holding on to their Mac of a very long period of time. We're doing well with both of those. In the -- and as Kevan alluded to in his comments, we're seeing school systems like the Kansas City Public Schools that are switching from Chromebooks and Windows PCs to the MacBook Neo. And I'm hearing anecdotally more and more of those kind of stories, both happening at the school system level and at the individual consumer level. And so we could not be happier with how things are going at the moment.
Michael Ng
Goldman Sachs Group, Inc., Research Division
Great. And for the second question, I wanted to ask about advertising within services. I think Apple introduced new inventory to ads on the App Store earlier this year. Has that new ad inventory on the App Store been a notable contributor to the services growth and outperformance in the quarter? And then could you talk more broadly about your ad strategy given the plans to also introduce ads to Maps this summer?
Kevan Parekh
Senior VP & CFO
Yes, Mike, it's Kevan. Thanks for the question. Yes, on advertising, we did see year-over-year growth in our advertising business. As you alluded to, we recently did introduce additional ads across the App Store search results to provide developers with more ways to drive downloads on platforms that users trust. And this summer, as you said, in the U.S. and Canada, Apple Maps will feature ads during key search and discovery moments, creating a new way for local businesses to reach customers and explore new places. But importantly, I think we believe it's possible to help businesses of all sizes grow via advertising while still delivering a great customer experience while also importantly, respecting people's fundamental right to privacy.
Operator
Our next question is from Wamsi Mohan with Bank of America.
Wamsi Mohan
BofA Securities, Research Division
Tim, you noted higher impact from memory as you look beyond the June quarter. Clearly, you guys have a lot of scale, supply chain efficiencies, relationships from a long time. As you think about product position relative to your competitors. So when you think about product position and pricing, relative to competition, do you think in such times of dislocation that Apple would be strategically more focused on share gain or where potentially you don't raise pricing and perhaps lower ends of the portfolio where your competitors are struggling or more focused on profitability? Like what's the right framework for us to think through as you enter that period? And I have a follow-up.
Timothy Cook
CEO & Director
Wamsi, we will look at a range of options with memory costs increasing. And so I really don't want to go beyond that at this point.
Wamsi Mohan
BofA Securities, Research Division
Okay. Okay, Tim. On -- as a follow-up here, how is Apple thinking about the broader monetization, maybe following Ben's question here in the Agentic AI world? So what parts of the stack do you think Apple will be focused on internally versus maybe leveraging your partners? I mean, we have some early looks into where you are developing relationships. But as we think longer term, do you think Apple will invest more? Where will Apple invest more heavily over the next several years? And is this at all related to your net cash comments in terms of perhaps building out more infrastructure as we enter an AI-centric world?
Timothy Cook
CEO & Director
We are clearly investing more. You can see that in the OpEx numbers. And if you click down on those a step deeper and look at the R&D area separate than SG&A, you'll find that R&D is even accelerating much higher than the company is. And so we are clearly investing. We're investing in products and services, and we see opportunities in both of those. And we could not be more excited about how the future is playing out.
Kevan Parekh
Senior VP & CFO
Yes. And I think, Wamsi, as we've talked about building on what Tim said, from the start, we said we believe AI is a really important investment area for Apple, and we're going to be doing that incrementally on top of what we normally invest in our product road map. And so I think I just wanted to reiterate that point as well.
Operator
Our next question is from Amit Daryanani with Evercore.
Amit Daryanani
Evercore ISI Institutional Equities, Research Division
I have 2 as well. I guess, first one, maybe just going back to the iPhone performance, which for a couple of quarters, you folks have had 20% plus growth despite the supply constraint. And I think the guide sort of implies the momentum will continue in June. I'd love for you folks to just maybe double-click and talk about what are the levers that's driving this sort of impressive iPhone growth despite the supply constraints? And then sort of what is the durability of this growth?
Timothy Cook
CEO & Director
Yes. If you look at it, it's the iPhone 17 family that's driving it. And that is, as you point out, is despite the supply constraints that we're experiencing. And it's the things that are driving people to the 17 are people love the design. People love the performance. They love the durability. They love the camera. They love center stage. And they love that Apple Intelligence is integrated across the platform. From where we're seeing the growth, it is amazing. We're seeing double-digit growth in the majority of the markets we track from the U.S. to Latin America to Greater China to Western Europe, to India to Japan to Southeast Asia. And we set a new March quarter record for upgraders as well. And what's driving all this is that the customer satisfaction for the 17 family in the U.S., as an example, is 99%. These numbers are just unheard of. And so we're thrilled with how things are going.
Amit Daryanani
Evercore ISI Institutional Equities, Research Division
Perfect. And then, Tim, I think we have you for one more earnings call, but I would really appreciate if you could kind of share a bit about the upcoming transition. You have historically, I think, talked about the advice that Steve gave you when you took over, and I might be paraphrasing this, but it was around, don't ask what I would do, just do the right thing. That's really been a big win, I think, for Apple and shareholders over the last 15 years. Would love to understand what advice are you giving John to help him build on Apple's strengths while shaping up the next chapter for the company.
Timothy Cook
CEO & Director
Well, I think Steve's advice to me lifted a huge burden. And so that advice did well for me over the 15 years. For John, I think my advice is that or what I told him is that one of the most important decisions he'll make is where to spend his time. And I would spend it where the greatest benefit to the company and the users are. And never forget the North Star for the company. We're about making the best products in the world that really enrich other people's lives. And if you keep focusing on that and make your decisions around that, it will produce a great business, and we'll be able to build more products and do it all over again.
Operator
Our next question is from David Vogt with UBS.
David Vogt
UBS Investment Bank, Research Division
Maybe, Tim, I want to go back to the supply chain for a second. I don't think I heard you state in your prepared remarks or in response to a question that the iPhone is constrained in the June quarter. So can you walk through kind of how you're thinking about your ability to secure not just SoC, but also memory? Are you thinking about using alternative sources of memory outside of sort of the traditional partners that you have? And just what's kind of driving that confidence that the iPhone isn't constrained given the amount of share it sounds like you're taking in that market? And then I have a follow-up as well.
Timothy Cook
CEO & Director
Yes. David, the current constraint for -- well, the constraint in the March quarter and the June quarter, the primary constraint is the availability of the advanced nodes our SoCs are produced on, not memory. And so I don't want to predict our ability to -- for supply and demand to match because if I look at it realistically, I think on the Mac Mini and the Max Studio, I believe it will take several months to reach supply-demand balance. And so we're not at the point where we're saying this is going to end anytime soon. And it's not because of a problem per se other than we just undercalled the demand. And there are lead times to this, as you well understand, and it takes a while to correct that. And the primary constraint from a product point of view in the -- or the majority of it for this quarter for the June quarter will be on the Mac. And it's Mac Mini, Mac Studio and the MacBook Neo. It's all of those.
David Vogt
UBS Investment Bank, Research Division
Great. And then maybe just on -- maybe on services real quick. Obviously, relatively strong gross margins yet again. Are we getting to a point given sort of the product mix within services? I know a lot of different offerings are growing double digits that we're sort of asymptotically getting to a level where we're seeing increasingly more challenging to scale that business from a profitability perspective? Or is there still sort of low-hanging fruit in terms of volume leverage in some of the offerings or maybe lower losses in some different categories that can continue to scale gross margin across the services base?
Kevan Parekh
Senior VP & CFO
Yes, David, it's Kevan. Thanks for the question. Look, as you know, our services portfolio contains a wide range of businesses that have different business models and profitability profiles and also are growing at different rates. So at any given time, right, the relative performance of those can impact the gross margin. This time, in particular, we look at the Q2 services margin, we talked about the fact that it increased 20 basis points sequentially. That's primarily driven by mix. And so again, I think it's hard to speculate how that evolves over time, but we're encouraged by what we're seeing. We do have some services that are improving in profitability as they gain scale. But again, I think we have a wide portfolio that has different characteristics and can grow at different rates at different times. But overall, we're encouraged by the overall trajectory that we've seen.
Operator
Our next question is from Samik Chatterjee with JPMorgan.
Samik Chatterjee
JPMorgan Chase & Co, Research Division
Tim, for my first question, last quarter, you did talk about Apple foundational models and sort of the two-pronged strategy there of the collaboration with Google as well as continuing to internally sort of work on your own models. Hoping you can sort of give us an update in terms of how you're able to balance those 2 priorities as well as do you feel like you need to double down and invest more to be able to balance those 2 priorities side by side? And I have a follow-up.
Timothy Cook
CEO & Director
Yes, it's a good question. We are investing more. You can see that in the OpEx numbers. And as I've mentioned before, the R&D, in particular, is -- has scaled rather significantly on a year-over-year basis. The collaboration with Google is going well. We're happy with where things are, and we're happy with the work that we're doing independently as well.
Samik Chatterjee
JPMorgan Chase & Co, Research Division
Okay. Great. And my follow-up for Kevan. Kevan, the sequential moderation in the product gross margin this year is relatively muted compared to what we've historically seen, at least over the last couple of years. Is it primarily mix? Or what was the -- maybe the FX tailwind as well? How would we sort of break it down in terms of what was different this year relative to what we typically see? And if you could sort of also clarify what the FX impact on gross margin was for the quarter?
Kevan Parekh
Senior VP & CFO
Sure, Samik. Let me start on products for Q2. Basically, products gross margin did decrease by 200 basis points sequentially, driven by a seasonal loss of leverage and higher memory costs, as Tim had alluded. If I zoom out, though, I think it's important just to look at what drove the overall company gross margin performance. And let me just give you a quick kind of rundown of that. If you look at our overall performance, right, our sequential gross margin impact was 110 basis points positively, and that was driven by favorable mix, lower tariff-related costs, and that was partly offset by seasonal loss of leverage and higher memory costs. And I did want to turn it over to Tim because we do want to provide some clarity around the lower tariff-related costs and just make a comment on that as well.
Timothy Cook
CEO & Director
Yes. Thanks, Kevan. For the March quarter, the gross margin of 49.3% did include the impact of tariff-related costs. However, tariffs in the March quarter versus the December quarter were lower because our -- we had lower product volume, as you know, sequentially from Q1 to Q2. And there was the full quarter benefit from a reduction in the IEEPA tariff rates as well as the reduced global tariff rate under Section 122. In terms of applying for a refund of tariffs paid, we're following the established processes, and we plan to reinvest any amount we receive back into U.S. innovation and advanced manufacturing. These would be new investments and would be in addition to our prior commitments in the U.S.
Kevan Parekh
Senior VP & CFO
And then one last point on your FX question. We really didn't see any sequential impact related to foreign exchange as a factor going from Q1 gross margin to Q2.
Suhasini Chandramouli
Director of Investor Relations
Operator, could we get the last question please?
Operator
We'll go ahead and take our last question from Aaron Rakers with Wells Fargo.
Aaron Rakers
Wells Fargo Securities, LLC, Research Division
Congrats on the quarter. I wanted to ask about a few of the end markets. I guess, particularly, Tim, if you could comment a little bit on what you're seeing specifically in China. I guess, from a competitive perspective, are you seeing advantages from supply constraints impacting some of your competitors? Any thoughts on the China market? And I do have a quick follow-up.
Timothy Cook
CEO & Director
Yes. We are thrilled with the performance in Greater China. The first half of the year grew at 33%. In the March quarter, revenue was up 28%. It's a quarterly revenue record for us. The performance is really driven by iPhone, which was also a March quarter record. If you look at the individual products, the iPhone was the top-selling model in urban China. The Mac Mini was the top-selling desktop in China and the MacBook Air was the top-selling laptop model. And so we're really doing well, pretty well across the board there. And I was over there in March. The traffic in our stores grew by double digit. We were celebrating Apple's 50th anniversary there, and it was just amazing to be a part of the community there. And so I'm really happy with how things have gone in the first half of this year.
Aaron Rakers
Wells Fargo Securities, LLC, Research Division
Yes. And then maybe I'll stick with a similar theme, kind of the same question on the India market. It seems like that continues to be a focal point on these last several quarterly conference calls. I mean, how are you seeing the market in India evolve around the base of iPhones and the opportunity of kind of a rising middle class, just the overall opportunity set in that large mobile market?
Timothy Cook
CEO & Director
Yes. I think it's a huge opportunity for us. We've been focused on this for a while. It's the second largest smartphone market in the world and the third largest PC market. And despite doing extremely well there for quite some time, we still have a modest share. And so I think there's -- that really speaks to the opportunity that we have. There are a lot of people moving into the middle class there, and we've got some great products for them, both currently and coming. And if you look at the majority of customers in all of our categories from the iPhone to the Mac to the iPad to the watch are new to that product there. And so it speaks very well to growing the installed base there. Net-net, I'm over the moon excited about India.
Suhasini Chandramouli
Director of Investor Relations
Thank you, Aaron. A replay of today's call will be available for 2 weeks on Apple podcast as a webcast on apple.com/investor and via telephone. The number for the telephone replay is (866) 583-1035, please enter confirmation code 2803309, followed by the pound sign. These replays will be available by approximately 5:00 p.m. Pacific Time today. Members of the press with additional questions can contact Josh Rosenstock at (408) 862-1142, and financial analysts can contact me, Suhasini Chandramouli with additional questions at (408) 974-3123. Thanks again for joining us today.
Operator
Once again, this does conclude today's conference. We do appreciate your participation.

"""

In [40]:
import pandas as pd

def parse_speaker_chunks(transcript: str) -> list:
    """Parse SA-style transcript using the participant directory as ground truth."""
    lines = [ln.rstrip() for ln in transcript.split("\n")]

    # 1. Build {name -> role} from participant sections
    directory = {}
    in_block = False
    for ln in lines:
        s = ln.strip()
        if s in ("Company Participants", "Conference Call Participants"):
            in_block = True
            continue
        if s in ("Presentation", "Question-and-Answer Session"):
            in_block = False
            continue
        if in_block and " - " in s:
            name, role = s.split(" - ", 1)
            directory[name.strip()] = role.strip()

    if not directory:
        raise ValueError("No participants found at top of transcript.")

    # 2. Find body start (after "Presentation")
    try:
        body_start = next(
            i for i, ln in enumerate(lines) if ln.strip() == "Presentation"
        ) + 1
    except StopIteration:
        body_start = 0
    body = lines[body_start:]

    # 3. Walk body, accumulate chunks
    chunks = []
    cur_speaker, cur_role, cur_text = None, None, []

    def flush():
        nonlocal cur_text
        if cur_speaker and cur_text:
            txt = "\n".join(cur_text).strip()
            if txt:
                chunks.append({"speaker": cur_speaker, "role": cur_role, "text": txt})
        cur_text = []

    i = 0
    while i < len(body):
        ln = body[i].strip()

        if ln in ("Question-and-Answer Session", "[Operator Instructions]"):
            i += 1
            continue

        if ln == "Operator":
            flush()
            cur_speaker, cur_role = None, None
            i += 1
            while i < len(body):
                nxt = body[i].strip()
                if nxt in directory:
                    j = i + 1
                    while j < len(body) and not body[j].strip():
                        j += 1
                    if j < len(body) and body[j].strip() == directory[nxt]:
                        break
                i += 1
            continue

        if ln in directory:
            j = i + 1
            while j < len(body) and not body[j].strip():
                j += 1
            if j < len(body) and body[j].strip() == directory[ln]:
                flush()
                cur_speaker = ln
                cur_role = directory[ln]
                i = j + 1
                continue

        if cur_speaker and ln:
            cur_text.append(ln)
        i += 1

    flush()
    return chunks


speaker_chunks  = parse_speaker_chunks(text)    # Q1
speaker_chunks2 = parse_speaker_chunks(text2)   # Q2

print(f"Q1: {len(speaker_chunks)} chunks")
print(f"Q2: {len(speaker_chunks2)} chunks")

Q1: 62 chunks
Q2: 43 chunks


In [41]:
print("Q1 first chunk:")
print(f"  speaker: {speaker_chunks[0]['speaker']}")
print(f"  role:    {speaker_chunks[0]['role']}")
print(f"  text:    {speaker_chunks[0]['text'][:120]}...")

print("\nQ2 first chunk:")
print(f"  speaker: {speaker_chunks2[0]['speaker']}")
print(f"  role:    {speaker_chunks2[0]['role']}")
print(f"  text:    {speaker_chunks2[0]['text'][:120]}...")

Q1 first chunk:
  speaker: Suhasini Chandramouli
  role:    Director of Investor Relations
  text:    Good afternoon, and welcome to the Apple Q1 Fiscal Year 2026 Earnings Conference Call. My name is Suhasini Chandramouli,...

Q2 first chunk:
  speaker: Suhasini Chandramouli
  role:    Director of Investor Relations
  text:    Good afternoon, and welcome to the Apple Q2 Fiscal Year 2026 Earnings Conference Call. My name is Suhasini Chandramouli,...


In [42]:
import pandas as pd

rows = []
for c in speaker_chunks:
    rows.append({**c, "quarter": "FQ1-2026", "filing_date": "2026-01-29"})
for c in speaker_chunks2:
    rows.append({**c, "quarter": "FQ2-2026", "filing_date": "2026-04-30"})

df_earnings = pd.DataFrame(rows)

# Drop very short chunks (filler like "Thanks, next question")
df_earnings = df_earnings[
    df_earnings["text"].str.split().str.len() >= 30
].reset_index(drop=True)

print(f"df_earnings: {len(df_earnings)} chunks")
print(df_earnings.groupby("quarter").size())

df_earnings: 89 chunks
quarter
FQ1-2026    49
FQ2-2026    40
dtype: int64


In [43]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Loading embedder (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

df_earnings = df_earnings.reset_index(drop=True)

print(f"Embedding {len(df_earnings)} chunks...")
chunk_embeddings = embedder.encode(
    df_earnings['text'].astype(str).tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"✓ Embeddings shape: {chunk_embeddings.shape}")

# Boilerplate filter
BOILERPLATE_KEYWORDS = [
    'forward-looking', 'safe harbor', 'private securities litigation',
    'investor relations contact', 'media contact', 'cautionary statement',
    'this press release contains', 'non-gaap',
]

def is_boilerplate(t):
    t = str(t).lower()
    if len(t) < 150:
        return True
    return sum(1 for kw in BOILERPLATE_KEYWORDS if kw in t) >= 2

df_earnings['is_boilerplate'] = df_earnings['text'].apply(is_boilerplate)
print(f"Boilerplate filtered: {df_earnings['is_boilerplate'].sum()} / {len(df_earnings)}")

# Quarter order (oldest → newest)
quarter_order = (
    df_earnings[['quarter', 'filing_date']]
    .drop_duplicates().sort_values('filing_date').reset_index(drop=True)
)
quarters_list = quarter_order['quarter'].tolist()
print(f"Quarters: {quarters_list}")

# Novelty: 1 - max cosine sim to any chunk in prior quarter
novelty_scores    = np.full(len(df_earnings), np.nan)
nearest_prior_idx = np.full(len(df_earnings), -1, dtype=int)

for i in range(1, len(quarters_list)):
    cur_q, prev_q = quarters_list[i], quarters_list[i - 1]
    cur_idx  = df_earnings.index[df_earnings['quarter'] == cur_q].to_numpy()
    prev_idx = df_earnings.index[df_earnings['quarter'] == prev_q].to_numpy()

    if len(cur_idx) == 0 or len(prev_idx) == 0:
        continue

    sim     = cosine_similarity(chunk_embeddings[cur_idx], chunk_embeddings[prev_idx])
    max_sim = sim.max(axis=1)
    argmax  = sim.argmax(axis=1)

    novelty_scores[cur_idx]    = 1.0 - max_sim
    nearest_prior_idx[cur_idx] = prev_idx[argmax]

df_earnings['novelty']           = novelty_scores
df_earnings['nearest_prior_idx'] = nearest_prior_idx

print("✓ Novelty computed.")

Loading embedder (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5298.83it/s]


Embedding 89 chunks...


Batches: 100%|██████████| 3/3 [00:00<00:00,  5.36it/s]

✓ Embeddings shape: (89, 384)
Boilerplate filtered: 0 / 89
Quarters: ['FQ1-2026', 'FQ2-2026']
✓ Novelty computed.


In [44]:
top = (
    df_earnings[
        (df_earnings['quarter'] == 'FQ2-2026') &
        (~df_earnings['is_boilerplate'])
    ]
    .nlargest(10, 'novelty')
    [['speaker', 'role', 'novelty', 'text']]
)

for _, row in top.iterrows():
    print(f"\n[novelty {row['novelty']:.3f}] {row['speaker']} ({row['role']})")
    print(f"  {row['text'][:250]}{'...' if len(row['text']) > 250 else ''}")


[novelty 0.646] Erik Woodring (Morgan Stanley, Research Division)
  All right. Awesome. And then Kevan, I'd love to maybe turn to you and kind of a surprise little announcement there talking about net cash neutral still a great path, but we're no longer providing this as a formal target. Could you maybe expand on tha...

[novelty 0.613] Kevan Parekh (Senior VP & CFO)
  Yes, sure, Erik. Thanks for the question. Yes, let me just kind of reiterate what we said, which is really kind of more of a comment on the capital structure. But our goal of net cash neutral has really served us well. It's been a valuable framework ...

[novelty 0.581] Kevan Parekh (Senior VP & CFO)
  And then one last point on your FX question. We really didn't see any sequential impact related to foreign exchange as a factor going from Q1 gross margin to Q2.

[novelty 0.580] Timothy Cook (CEO & Director)
  Well, I think Steve's advice to me lifted a huge burden. And so that advice did well for me over the 15 years. F

In [45]:
import os
import getpass

ANTHROPIC_API_KEY = os.environ.get("")
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key (sk-ant-...): ").strip()

assert ANTHROPIC_API_KEY.startswith("sk-ant-"), \
    "That doesn't look like an Anthropic key — should start with 'sk-ant-'"

TOP_N_FOR_LLM = 10
MODEL = "claude-sonnet-4-5"

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")

✓ API key loaded
✓ Using model: claude-sonnet-4-5


In [46]:
def build_transition_payload(df, cur_q, prev_q, top_n=TOP_N_FOR_LLM):
    q_df = df[
        (df['quarter'] == cur_q)
        & (~df['is_boilerplate'])
        & (df['novelty'].notna())
    ].sort_values('novelty', ascending=False).head(top_n)

    pairs = []
    for _, row in q_df.iterrows():
        prior = df.loc[row['nearest_prior_idx']]
        pairs.append({
            'novelty': round(float(row['novelty']), 3),
            'novel_chunk': str(row['text']).strip(),
            'closest_prior_chunk': str(prior['text']).strip(),
        })
    return pairs

test_pairs = build_transition_payload(df_earnings, 'FQ2-2026', 'FQ1-2026')
print(f"Built {len(test_pairs)} pairs for FQ1-2026 → FQ2-2026")
if test_pairs:
    print(f"First pair novelty: {test_pairs[0]['novelty']}")
    print(f"First novel chunk: {test_pairs[0]['novel_chunk'][:150]}...")

Built 10 pairs for FQ1-2026 → FQ2-2026
First pair novelty: 0.646
First novel chunk: All right. Awesome. And then Kevan, I'd love to maybe turn to you and kind of a surprise little announcement there talking about net cash neutral stil...


In [47]:
SYSTEM_PROMPT = """You are a financial text analyst examining quarter-over-quarter changes in earnings press releases.

You will receive pairs of text chunks. In each pair:
- "novel_chunk" is a passage from the current quarter's earnings release
- "closest_prior_chunk" is the most semantically similar passage from the previous quarter's release
- "novelty" is a score from 0 to 1 where higher values mean less similar

Your job: identify what NEW CONTENT appears in the current quarter that wasn't in the previous quarter. Classify each distinct new theme into one of these categories:

- new_product: A product, service, SKU, or offering mentioned that wasn't in the prior quarter
- new_metric: A financial or operating metric being reported that wasn't reported before, OR a new segment breakdown
- new_risk: A risk factor, challenge, or headwind mentioned that wasn't in the prior quarter
- new_segment: A business segment, customer type, or geography being discussed that wasn't before
- new_framing: The same topic as last quarter but described in a substantively different way
- other: New content that doesn't fit the above categories

STRICT RULES:
1. Every theme you identify MUST include a verbatim quote from the novel_chunk text. Put quotes in a "quotes" field.
2. Do NOT comment on tone, confidence, hedging, sentiment, or how cautious/bullish the language sounds. Only describe what content is new.
3. Do NOT speculate about why the change happened or what it means for the business. Only describe what is different.
4. If a "novel_chunk" is actually similar to its prior chunk (just reworded), say so in the "notes" field and mark it as new_framing with low confidence.
5. Consolidate themes across multiple chunks — if three chunks all introduce the same new product, output ONE theme with all three quotes.

Output format: valid JSON only, no other text. Schema:
{
  "themes": [
    {
      "category": "new_product" | "new_metric" | "new_risk" | "new_segment" | "new_framing" | "investment" | "other",
      "label": "Short descriptive label (3-8 words)",
      "description": "1-2 sentence neutral description of what's new",
      "quotes": ["verbatim phrase from novel_chunk", "another verbatim phrase"],
      "confidence": "high" | "medium" | "low",
      "notes": "any caveats about whether this is genuinely new"
    }
  ]
}"""

def build_user_message(prev_q, cur_q, pairs):
    lines = [
        f"Transition: {prev_q} → {cur_q}",
        f"Number of pairs: {len(pairs)}",
        "",
        "Pairs (sorted by novelty, highest first):",
        "",
    ]
    for i, p in enumerate(pairs, 1):
        lines.append(f"--- Pair {i} (novelty={p['novelty']}) ---")
        lines.append(f"NOVEL [{cur_q}]:")
        lines.append(p['novel_chunk'])
        lines.append("")
        lines.append(f"CLOSEST PRIOR [{prev_q}]:")
        lines.append(p['closest_prior_chunk'])
        lines.append("")
    return "\n".join(lines)

print("✓ Prompt and user-message builder defined")

✓ Prompt and user-message builder defined


In [48]:
#second version - have system decide what metrics to use.
SYSTEM_PROMPT = """You are a financial text analyst examining quarter-over-quarter changes in earnings calls.

You will receive pairs of text chunks. In each pair:
- "novel_chunk" is a passage from the current quarter
- "closest_prior_chunk" is the most semantically similar passage from the previous quarter
- "novelty" is a score from 0 to 1 — higher = less similar to anything in the prior quarter

The novelty score is your guide:
- novelty > 0.7: chunk likely contains genuinely new content (new product, new metric, new framing, or new event)
- novelty 0.4–0.7: partial overlap — chunk likely reframes or expands on prior content, OR contains some new content alongside familiar content
- novelty < 0.4: chunk is mostly recurring content (same topic, similar wording)

For each chunk pair, identify what NEW CONTENT (if any) appears in the novel_chunk that wasn't in closest_prior_chunk. Classify each distinct new theme into one of these categories:

- new_product: A product, service, or offering not in the prior quarter
- new_metric: A financial or operating metric reported that wasn't reported before, OR a new breakdown of an existing metric
- new_risk: A risk, challenge, or headwind not previously mentioned
- new_segment: A business segment, customer type, or geography not discussed before
- new_framing: Same topic as before, but described in a substantively different way
- other: New content not fitting the above

STRICT RULES:
1. Let the novelty score guide your interpretation. A high-novelty chunk should produce a confident theme; a low-novelty chunk may produce no themes (just say so in notes).
2. Every theme MUST include a verbatim quote from novel_chunk in a "quotes" field.
3. Do NOT comment on tone, sentiment, or hedging.
4. Do NOT speculate about implications.
5. If three chunks introduce the same theme, consolidate them into ONE theme with all quotes.

Output: valid JSON only. Schema:
{
  "themes": [
    {
      "category": "new_product" | "new_metric" | "new_risk" | "new_segment" | "new_framing" | "other",
      "label": "3-8 words",
      "description": "1-2 sentence neutral description of what's new",
      "quotes": ["verbatim phrase", "another verbatim phrase"],
      "novelty_interpretation": "what the novelty score suggests about this chunk",
      "confidence": "high" | "medium" | "low",
      "notes": "any caveats"
    }
  ]
}"""

In [49]:
import os
import getpass
import anthropic
import json

# Ask for the key (or use env var if already set)
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key (sk-ant-...): ").strip()

assert ANTHROPIC_API_KEY.startswith("sk-ant-"), \
    "That doesn't look like an Anthropic key — should start with 'sk-ant-'"

MODEL = "claude-sonnet-4-5"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def summarize_transition(prev_q, cur_q, pairs):
    if len(pairs) == 0:
        return {"themes": [], "error": "no pairs available"}

    user_msg = build_user_message(prev_q, cur_q, pairs)

    response = client.messages.create(
        model=MODEL,
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}],
    )

    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as e:
        return {"themes": [], "error": f"JSON parse failed: {e}", "raw_response": raw[:500]}

    parsed['_meta'] = {
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
        'pairs_sent': len(pairs),
    }
    return parsed


# Run the comparison
transition_summaries = {}
cur_q = 'FQ2-2026'
prev_q = 'FQ1-2026'

print(f"Comparing {prev_q} → {cur_q}...")
pairs = build_transition_payload(df_earnings, cur_q, prev_q)
print(f"  Sending {len(pairs)} pairs to Claude...")
result = summarize_transition(prev_q, cur_q, pairs)
transition_summaries[f"{prev_q}->{cur_q}"] = result

if 'error' in result:
    print(f"  ✗ error: {result['error']}")
else:
    meta = result.get('_meta', {})
    print(f"  ✓ got {len(result['themes'])} themes "
          f"({meta.get('input_tokens', '?')} in / {meta.get('output_tokens', '?')} out tokens)")

Comparing FQ1-2026 → FQ2-2026...
  Sending 10 pairs to Claude...
  ✓ got 6 themes (4450 in / 1417 out tokens)


In [50]:
def print_themes(summary, transition_label):
    if 'error' in summary:
        print(f"Error: {summary['error']}")
        return

    themes = summary.get('themes', [])
    print(f"\n{'='*72}")
    print(f"  {transition_label}: {len(themes)} themes identified")
    print(f"{'='*72}")

    by_cat = {}
    for t in themes:
        by_cat.setdefault(t['category'], []).append(t)

    category_order = ['new_product', 'new_segment', 'new_metric',
                      'new_risk', 'new_framing', 'other','investment']

    for cat in category_order:
        if cat not in by_cat:
            continue
        print(f"\n── {cat.upper()} ──")
        for t in by_cat[cat]:
            print(f"\n  • {t['label']}  [{t.get('confidence', '?')}]")
            print(f"    {t['description']}")
            for q in t.get('quotes', []):
                print(f"    → \"{q}\"")
            if t.get('notes'):
                print(f"    notes: {t['notes']}")

print_themes(transition_summaries['FQ1-2026->FQ2-2026'], 'FQ1-2026 → FQ2-2026')

# Flatten to DataFrame for downstream use
rows = []
for key, summary in transition_summaries.items():
    if 'error' in summary:
        continue
    prev_q, cur_q = key.split('->')
    for t in summary.get('themes', []):
        rows.append({
            'prev_quarter': prev_q,
            'current_quarter': cur_q,
            'category': t['category'],
            'label': t['label'],
            'description': t['description'],
            'quotes': ' | '.join(t.get('quotes', [])),
            'confidence': t.get('confidence'),
            'notes': t.get('notes', ''),
        })

df_themes = pd.DataFrame(rows)
df_themes


  FQ1-2026 → FQ2-2026: 6 themes identified

── NEW_PRODUCT ──

  • MacBook Neo Supply Constraints and Customer Response  [high]
    Management discusses supply constraints on a product called MacBook Neo, noting stronger-than-expected demand and success with school systems like Kansas City Public Schools switching from Chromebooks and Windows PCs.
    → "Right now, we're supply constrained on the MacBook Neo"
    → "we undercalled the level of enthusiasm that we'd be with it"
    → "we're seeing school systems like the Kansas City Public Schools that are switching from Chromebooks and Windows PCs to the MacBook Neo"
    notes: MacBook Neo appears to be either a new product name or possibly a product not discussed in the prior quarter transcripts provided.

── NEW_METRIC ──

  • FX Had No Sequential Gross Margin Impact  [high]
    Management explicitly states that foreign exchange had no sequential impact on gross margin moving from Q1 to Q2, providing clarity on FX effects.
    → "We 

,prev_quarter,current_quarter,category,label,description,quotes,confidence,notes
0,FQ1-2026,FQ2-2026,new_framing,Net Cash Neutral No Longer Formal Target,Apple announces it is no longer providing net ...,we're no longer providing this as a formal tar...,high,This is a clear policy announcement not presen...
1,FQ1-2026,FQ2-2026,new_metric,FX Had No Sequential Gross Margin Impact,Management explicitly states that foreign exch...,We really didn't see any sequential impact rel...,high,This is a specific data point not mentioned in...
2,FQ1-2026,FQ2-2026,new_product,MacBook Neo Supply Constraints and Customer Re...,Management discusses supply constraints on a p...,"Right now, we're supply constrained on the Mac...",high,MacBook Neo appears to be either a new product...
3,FQ1-2026,FQ2-2026,other,CEO Transition Advice to Successor,Tim Cook shares advice given to his successor ...,my advice is that or what I told him is that o...,high,This represents new disclosure about leadershi...
4,FQ1-2026,FQ2-2026,new_framing,Agentic Smartphone and Form Factor Discussion,Question raised about 'Agentic smartphone' con...,there's been some commentary around an Agentic...,medium,This represents a new framing of AI questions ...
5,FQ1-2026,FQ2-2026,new_framing,R&D Acceleration Higher Than Company Average,Management highlights that R&D spending is acc...,if you click down on those a step deeper and l...,high,This provides new granularity on spending comp...
